In [ ]:
import pickle
import os
import itertools
import scipy
import csv
import pandas as pd
import numpy as np
import scanpy as sc
import scvelo as scv
import seaborn as sns
import gseapy as gp
import anndata as ad
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import matplotlib.colors as mcolors
from matplotlib.backends.backend_pdf import PdfPages
from matplotlib.pyplot import rc_context
from matplotlib.colors import LogNorm, Normalize
from scipy import sparse
from scipy import stats
from copy import copy
from scipy.stats import norm
from sklearn.neighbors import KernelDensity
from typing import Tuple
from pathlib import Path
from matplotlib.colors import LinearSegmentedColormap, Normalize

In [ ]:
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)
sc.settings.verbosity = 0

# Load helpful objects functions

In [ ]:
all_datasets = ['E11_iScience_R1', 'E12_DevCell', 'E12_CellReports_R1', 'E13_CellReports', 'E12_Science', 'E11lungR1', 'E11lungR2', 'E11lungR3', 'E11lungR4', 'E11lungR5', 'E12lungR2', 'E12lungR3', 'E12lungR4', 'E12lungR5', 'E13lungR1', 'E13lungR2', 'E13lungR3', 'E13lungR4']
dbit_datasets = ['E11lungR1', 'E11lungR2', 'E11lungR3', 'E11lungR4', 'E11lungR5', 'E12lungR2', 'E12lungR3', 'E12lungR4', 'E12lungR5', 'E13lungR1', 'E13lungR2', 'E13lungR3', 'E13lungR4']
ref_datasets = ['E11_iScience_R1', 'E12_DevCell', 'E12_CellReports_R1', 'E13_CellReports', 'E12_Science']
timepoints = ['E11.5', 'E12.5', 'E13.5']
cellstates = ['MesenchymeProgenitor', 'SmoothMuscleProgenitor', 'SmoothMuscle', 'MatrixMesenchyme', 'Precartilage', 'Pericyte',
              'Mesothelium', 'Epithelial(Distal)', 'Epithelial(Proximal)', 'VascularEndothelial',
              'Immune', 'Blood', 'Neuron', 'Heart']
epi_subcluster_cellstates = ['Distal', 'DistalIntermediate', 'ProximalIntermediate', 'Proximal']
mes_subcluster_cellstates = ['MP1', 'MP2', 'MP3', 'SmoothMuscleProgenitor', 'SmoothMuscle', 'MatrixMesenchyme', 'MatrixMesenchymeProgenitor', 'Precartilage', 'Pericyte']

# figure out color scheme
clusterColors = {'Neuron': 'DarkGreen',
                 'MatrixMesenchyme': 'DarkRed',
                 'MesenchymeProgenitor': 'RoyalBlue',
                 'Epithelial(Distal)': 'SkyBlue',
                 'Epithelial(Proximal)': 'Red',
                 'DbitEpithelialBridge': 'Cyan',
                 'SmoothMuscleProgenitor': 'Magenta',
                 'SmoothMuscle': 'Purple',
                 'VascularEndothelial': 'gold',
                 'Mesothelium': 'limegreen',
                 'Precartilage': 'darkgoldenrod',
                 'Immune': 'DarkOrange',
                 'Heart': 'Gray',
                 'Blood': 'tan',
                 'Pericyte': 'lightpink'}

epiSubclusterColors = {'Proximal': 'Red',
                       'ProximalIntermediate':'darkred',
                       'DistalIntermediate':'darkblue',
                       'Intermediate': 'Green',
                       'Distal':'SkyBlue'}

mesSubclusterColors = {'MP1': 'RoyalBlue',
                       'MP2':'lawngreen',
                       'MP3':'Orange',
                       'MP4':'salmon',
                       'MatrixMesenchyme':'DarkRed',
                       'MatrixMesenchymeProgenitor': 'Red',
                       'SmoothMuscleProgenitor': 'Magenta',
                       'SmoothMuscle':'Purple',
                       'Pericyte': 'lightpink',
                       'Precartilage':'darkgoldenrod'}

phaseDict = {'G1': 'Blue', 'G2M': 'Orange', 'S': 'Green'}

greens = copy(mpl.cm.Greens)
greens.set_under("lightgray")
reds = copy(mpl.cm.Reds(np.linspace(0,1,40)))
reds = mpl.colors.ListedColormap(reds[5:])
reds.set_under("lightgray")

# Add silver as the lower color for the expression colormap
colors = ["silver", "red"]  # Colors for the gradient
expr_cmap = LinearSegmentedColormap.from_list("silver_to_red", colors)
expr_cmap.set_over("red") 
norm = Normalize(vmin=0, vmax=10)

In [ ]:
def plot_spatial_groups(adata, 
                        meta_col,
                        background_adata = False, 
                        palette = False, 
                        show = True, 
                        output_loc = False, 
                        name = 'spatial_plot', 
                        size = 20):
    '''
    input
        - adata: An anndata object. Assumes 14 dbit datasets (split by 'orig.ident'), labeled as dbit in the 'tech' column. Assumes there is an x_loc and y_loc col
        - background_adata: A second anndata object that contains all the cells to plot in the background (if adata is a subset).
            if not provided then it will take adata as the background adata.
        - meta_col: A column in the .obs table or gene in the .var table
        - palette: A dictionary that contains a desired color for each value in the meta_col column or a cmap for a quantitative meta_col.
        - show: True or False on whether to show the plot. Default is False
        - output_loc: Whether or not to save the plot and what folder location. Default is False. 
        - name: The name to give the output pdf if saving the plot
        - size: The dot size for the scatter plot
    returns
        creates spatial scatter plots for the 14 lung spatial datasets. Options to save and show the plot
    '''
    # check if the output_loc folder exists, return an error if not
    if output_loc != False:
        if not os.path.exists(output_loc):
            print('Error: output_loc folder does not exist')
            return
        
    # check if meta_col exists, returns an error if not
    if meta_col not in adata.obs.columns and meta_col not in adata.var.index:
        print('Error: meta_col not in .obs or .var tables')
        return
    
    if type(background_adata) == type(False):
        background_adata = adata
    
    # Make a temp object to work with
    temp_adata = adata.copy()
    
    # Add the gene expr values to the obs table if plotting a gene
    if meta_col in adata.var.index:
        temp_adata.obs[meta_col] = temp_adata[:, meta_col].X.todense()
    
    timepoints = ['E11.5', 'E12.5', 'E13.5']
    fig, axs = plt.subplots(3, 5, figsize = (35, 21))
    row_count = 0
    for time in timepoints:
        datasets = temp_adata.obs[(temp_adata.obs['timepoint'] == time) & (temp_adata.obs['tech'] == 'DBiT_seq')]['orig.ident'].unique()

        col_count = 0
        for exp in datasets:
            temp_ax = axs[row_count, col_count]
            col_count += 1
            
            # Plot full structure in gray from the background adata
            all_cell_locs = background_adata.obs[(background_adata.obs['orig.ident'] == exp)].index
            x = []
            y = []

            for i in all_cell_locs:
                y.append(background_adata.obs.loc[i, 'y_loc'])
                x.append(background_adata.obs.loc[i, 'x_loc'])
            # Use silver with a default alpha to match the lower bound of the expression plots
            temp_ax.scatter(x, y, color = 'silver', s = size)
        
            # plot the quantitative variable
            temp_locs = temp_adata.obs[(temp_adata.obs['orig.ident'] == exp)].index
            
            if type(temp_adata.obs[meta_col][0]) != str:
                x = [0]
                y = [0]
                z = [temp_adata.obs[temp_adata.obs['tech'] == 'DBiT_seq'][meta_col].max()]
                for i in temp_locs:
                    z.append(temp_adata.obs.loc[i, meta_col])
                    y.append(temp_adata.obs.loc[i, 'y_loc'])
                    x.append(temp_adata.obs.loc[i, 'x_loc'])
                if palette != False:
                    colorbar = temp_ax.scatter(x, y, c = z, s = size, cmap = palette)
                else:
                    colorbar = temp_ax.scatter(x, y, c = z, s = size, cmap = 'viridis')
            else:
                labels = ['background']
                state_list = temp_adata.obs[meta_col].unique().tolist()
                for state in state_list:
                    temp_locs = temp_adata.obs[(temp_adata.obs['orig.ident'] == exp) & (temp_adata.obs[meta_col] == state)].index

                    x = [0]
                    y = [0]
                    s = [size]
                    for i in temp_locs:
                        y.append(temp_adata.obs.loc[i, 'y_loc'])
                        x.append(temp_adata.obs.loc[i, 'x_loc'])
                        s.append(size)
                    labels.append(state)

                    if palette != False:
                        temp_ax.scatter(x, y, s = size, color = palette[state])
                    else:
                        temp_ax.scatter(x, y, s = size)
                    if row_count == 0 and col_count == 5:
                        temp_ax.legend(labels = labels, bbox_to_anchor = (1.02, 1))
            temp_ax.set_title('{}_{}'.format(exp, meta_col))
            temp_ax.set_xlabel('X Pos')
            temp_ax.set_ylabel('Y Pos')
            temp_ax.set_xlim(0, 52)
            temp_ax.set_ylim(0, 52)

        row_count += 1
    if type(temp_adata.obs[meta_col][0]) != str:
        fig.colorbar(colorbar, ax = axs[2, 4])
    if output_loc != False:
        plt.savefig(output_loc + name, dpi = 300)
    if show:
        plt.show()
    plt.close()

In [ ]:
def plot_volcano(temp_df_loc, 
                 upName,
                 downName,
                 genes = [],
                 annotate_genes = [],
                 palette = {},
                 show = True, 
                 output_loc = False, 
                 name = 'volcano_plot',
                 size = (5, 5)):
    '''
    input
        - temp_df_loc: a file location for the DE csv file from calculateDE. Assumed to have a pval_adj, and manual_LogFC column for each gene.
        - upName: a string name for the positive side of the DE comparison. Assumed to have a {}_avg_expr column in the temp_df
        - downName: a string name for the positive side of the DE comparison. Assumed to have a {}_avg_expr column in the temp_df
        - genes: a list of genes to plot in the volcano plot. If none are provided then all will be used.
        - annotate_genes: a list of genes to annotate. If none are provided then the top 10 scoring genes in both directions will be annotated
        - colorDict: a dictionary of colors to label significant genes. Assumed to contain keys matching upName and downName. If none are provided, colors will be selected
        - show: A boolean plot the volcano or not. Default is False
        - output_loc: a string of the folder location for saving the plot. Default is False to not save the plot
        - name: a string to name the saved file
        - size: a tuple of the x and y sizes for the figure. Default is (5, 5).
    returns
        Creates a volcano plot of the differentially expressed genes
    '''
    # Plot a volcano plot of Distal vs DI
    temp_df = pd.read_csv(temp_df_loc, index_col = 0)
    
    # add pseudolowest pvals for any that are too low for the computer
    temp_df['pvals_adj_2'] = temp_df['pvals_adj']
    for gene in temp_df[temp_df['pvals_adj'] == 0].index:
        temp_df.loc[gene, 'pvals_adj_2'] = temp_df[temp_df['pvals_adj'] > 0]['pvals_adj'].min()

    if len(genes) == 0:
        genes = temp_df.index.tolist()
    else:
        pass

    
    # subset to genes in the genes list
    temp_df = temp_df[temp_df.index.isin(genes)]
    
    # Checks if genes to annotate were given
    if len(annotate_genes) == 0:
        # grab the top 10 genes on both sides
        annotate_genes = temp_df[(temp_df['pvals_adj'] < 0.05) & (temp_df['manual_LogFC'] > 1)][:10].index.tolist() + temp_df[(temp_df['pvals_adj'] < 0.05) & (temp_df['manual_LogFC'] < -1)][-10:].index.tolist()
    else:
        pass

    # Checks if a colorDict was given
    if upName in palette.keys() and downName in palette.keys():
        pass
    else:
        palette = {upName: 'red', downName: 'royalblue'}
    
    fig, ax = plt.subplots(figsize = size)

    # Plot the volcano plot of the differential expression using manual LogFC
    x = temp_df[(temp_df['{}_avg_expr'.format(upName)] != 0) & (temp_df['{}_avg_expr'.format(downName)] != 0)]['manual_LogFC'].values
    y = -np.log10(temp_df[(temp_df['{}_avg_expr'.format(upName)] != 0) & (temp_df['{}_avg_expr'.format(downName)] != 0)]['pvals_adj_2']).values
    plt.scatter(x = x, y = y, alpha = 0.3, s = 5, color = 'darkgray')
    
    # highlight genes that are higher than 1 logfoldchange and higher than 1.3 -log10 adjustedpval
    x_up = []
    y_up = []
    x_down = []
    y_down = []
    for count, val in enumerate(x):
        if x[count] > 1 and y[count] > 1.3:
            x_up.append(x[count])
            y_up.append(y[count])
        if x[count] < -1 and y[count] > 1.3:
            x_down.append(x[count])
            y_down.append(y[count])
    plt.scatter(x = x_up, y = y_up, s = 8, color = palette[upName])
    plt.scatter(x = x_down, y = y_down, s = 10, color = palette[downName])

    # Annotate the genes
    for gene in annotate_genes:
        # Check that the annotation gene is present in the list of genes
        if gene in genes:
            x = temp_df.loc[gene, 'manual_LogFC']
            y = -np.log10(temp_df.loc[gene, 'pvals_adj_2'])
            plt.text(x, y + 0.5, gene) 
        else:
            print('WARNING: {} not present in genes'.format(gene))
    
    ax.axhline(1.3, zorder = 0, c = 'gray', lw = 1, ls = '--')
    ax.axvline(1, zorder = 0, c = 'gray', lw = 1, ls = '--')
    ax.axvline(-1, zorder = 0, c = 'gray', lw = 1, ls = '--')
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    plt.xlabel('{} / {} (LogFC)'.format(upName, downName))
    plt.ylabel('-log10(adj_pval)')
    plt.title('{} vs {} - DE'.format(upName, downName))
    if output_loc != False:
        plt.savefig(output_loc + name, dpi = 300)
    if show:
        plt.show()
    plt.close()

In [ ]:
def calculateDE(adata, column, comparisons, output_loc, sampleName):
    '''
    input:
        adata - a scanpy object to run the analysis on
        column - a column in the adata.obs table that will be used to separate the data into 2 categories
        comparisons - a tuple that contains the 2 categories from the column to separate the data. The second is the ref name
        output_loc - a string that points to a valid folder to save the files
        name - a name for the 
    output:
        a volcano plot of the resulting gene expression data (logfc vs -log10pval)
        a plot of the scanpy outputs from the differential expression (top genes by score)
        a table that stores the differential expression values for each gene
    '''
    # check if the output loc folder exists, return none if not
    if not os.path.isdir(output_loc):
        print('Output_loc is not a folder')
        return None
    
    # check that the plots and gene lists folders exist, make them if not
    if not os.path.isdir(output_loc + 'plots/'):
        os.mkdir(output_loc + 'plots/')
    if not os.path.isdir(output_loc + 'gene_lists/'):
        os.mkdir(output_loc + 'gene_lists/')
    
    # set the scanpy figure directory to output_loc
    sc.settings.figdir = output_loc + 'plots/'
    sc.settings.verbosity = 0
    
    upName = comparisons[0]
    downName = comparisons[1]

    # subset the adata to include only the rows that have the comparisons
    temp_adata = adata[(adata.obs[column].isin(comparisons))].copy()
    
    sc.tl.rank_genes_groups(temp_adata, column, method='wilcoxon', groups = [upName], reference = downName, key_added = "wilcoxon", pts = True, use_raw = False)
    sc.pl.rank_genes_groups(temp_adata, n_genes=25, sharey=False, key="wilcoxon", save = '_{}_{}_vs_{}'.format(sampleName, upName, downName), use_raw = False, show = False)
    
    # record the information from the logfc table
    names = [i[0] for i in temp_adata.uns['wilcoxon']['names'].tolist()]
    scores = [i[0] for i in temp_adata.uns['wilcoxon']['scores'].tolist()]
    pvals = [i[0] for i in temp_adata.uns['wilcoxon']['pvals'].tolist()]
    pvals_adj = [i[0] for i in temp_adata.uns['wilcoxon']['pvals_adj'].tolist()]
    logfcs = [i[0] for i in temp_adata.uns['wilcoxon']['logfoldchanges'].tolist()]
    
    pts = temp_adata.uns['wilcoxon']['pts']
    
    # save it to a df
    temp_df = pd.DataFrame(zip(names, scores, pvals, pvals_adj, logfcs), columns = ['gene', 'scores', 'pvals', 'pvals_adj', 'logfoldchanges'])

    # record the avg and frac expr for TLS and TLSCL for each gene
    genes = temp_df['gene']
    #return temp_adata, temp_df
    temp_df[upName + '_avg_expr'] = temp_adata[temp_adata.obs[column] == upName][:, genes].X.mean(axis = 0).tolist()[0]
    temp_df[upName + '_frac_expr'] = pts.loc[temp_df['gene'], upName].values

    temp_df[downName + '_avg_expr'] = temp_adata[temp_adata.obs[column] == downName][:, genes].X.mean(axis = 0).tolist()[0]
    temp_df[downName + '_frac_expr'] = pts.loc[temp_df['gene'], downName].values
    
    temp_df['frac_diff'] = abs(temp_df[upName + '_frac_expr'] - temp_df[downName + '_frac_expr'])
    temp_df['-log10pval_adj'] = -np.log10(temp_df['pvals_adj'])
    temp_df['manual_LogFC'] = np.log2(temp_df[upName + '_avg_expr'] + 0.01) - np.log2(temp_df[downName + '_avg_expr'] + 0.01) 
    
    # save the df
    temp_df.to_csv(output_loc + 'gene_lists/{}_{}_{}_wilcox_DE.csv'.format(sampleName, upName, downName), index = False)

    # Plot and save a volcano plot
    plot_volcano(temp_df_loc = output_loc + 'gene_lists/{}_{}_{}_wilcox_DE.csv'.format(sampleName, upName, downName), 
                 upName = upName,
                 downName = downName,
                 show = False, 
                 output_loc = output_loc + 'plots/', 
                 name = '{}_{}_vs_{}_volcano_man_LogFC.pdf'.format(sampleName, upName, downName),
                 size = (5, 5))

In [ ]:
def gmt_to_decoupler(pth: Path) -> pd.DataFrame:
    """
    Parse a gmt file to a decoupler pathway dataframe.
    """
    from itertools import chain, repeat

    pathways = {}

    with Path(pth).open("r") as f:
        for line in f:
            name, _, *genes = line.strip().split("\t")
            pathways[name] = genes

    return pd.DataFrame.from_records(
        chain.from_iterable(zip(repeat(k), v) for k, v in pathways.items()),
        columns=["geneset", "genesymbol"],
    )

In [ ]:
def runGSEApy(DE_file, gmt_file, output_folder, name, genes = None, gene_set_min = 15, gene_set_max = 500):
    '''
    run GSEA using the gseapy package on the resulting files from the calculate DE function above
    
    input:
        DE_file - the location of a DE file created by the above script
        gmt_file - the location of a gmt file that contains gene sets and genes. Assumes that the gene sets are not all uppercase
        output_loc - the location to save the GSEA results
        name - a label for the results tables (cell state, timepoint, DE comparisons for example)
        genes - a list of genes to perform the GSEA on. If none it will use all the genes. Default is None
    output:
        A table that contains the enrichment score, normalized enrichment score, and adjusted pval for each gene set
    '''
    GSEA_temp = pd.read_csv(DE_file, index_col = 0)
    rankfile_loc = output_folder + '/rankfiles/{}_rank.rnk'.format(name)
    output_loc = output_folder + '{}/'.format(name)
    
    GSEA_temp['gene'] = GSEA_temp.index
    
    # check to see if a folder exists to put the GSEA results
    if not os.path.exists(output_folder):
        os.mkdir(output_folder)
        
    if not os.path.exists(output_folder + 'rankfiles/'):
        os.mkdir(output_folder + 'rankfiles/')
        
    if not os.path.exists(output_loc):
        os.mkdir(output_loc)

    drop_labels = []
    # drop all columns but score and gene name
    for i in GSEA_temp.columns:
        if i != 'gene':
            if i != 'scores':
                drop_labels.append(i)
    GSEA_temp.drop(columns = drop_labels, inplace = True)
    
    if type(genes) != type(None):
        GSEA_temp = GSEA_temp[GSEA_temp['gene'].isin(genes)]
        
    GSEA_temp.drop(columns = 'gene', inplace = True)
    GSEA_temp.to_csv(rankfile_loc, sep = '\t', header = False)

    pre_res = gp.prerank(rnk = rankfile_loc,
             gene_sets = gmt_file,
             threads = 4,
             min_size = gene_set_min,
             max_size = gene_set_max,
             permutation_num = 1000, # reduce number to speed up testing
             outdir=output_loc,
             seed = 6,
             verbose = True,
            )

In [ ]:
def runDecoupler(DE_file, gmt_file, output_loc, name, genes = None, gene_set_min = 15, gene_set_max = 500):
    '''
    run GSEA using the decoupler package on the resulting files from the calculate DE function above
    
    input:
        DE_file - the location of a DE file created by the above script
        gmt_file - the location of a gmt file that contains gene sets and genes. Assumes that the gene sets are not all uppercase
        output_loc - the location to save the GSEA results
        name - a label for the results tables (cell state, timepoint, DE comparisons for example)
        genes - a list of genes to perform the GSEA on. If none it will use all the genes. Default is None
        gene_set_min - A threshold to filter gene sets below a certain size. Default is 15
        gene_set_max - A threshold to filter gene sets above a certain size. Default is 500
    output:
        A table that contains the enrichment score, normalized enrichment score, and adjusted pval for each gene set
    '''
    # open the GSEA gmt file
    hallmark = gmt_to_decoupler(gmt_file)
    #hallmark

    # Filtering genesets that are too small
    geneset_size = hallmark.groupby("geneset").size()
    gsea_genesets = geneset_size.index[(geneset_size > gene_set_min) & (geneset_size < gene_set_max)]

    GSEA_temp = pd.read_csv(DE_file, index_col = 0)
    rankfile_loc = output_loc + 'rankfiles/{}_vs_rest_rank.rnk'.format(name)
    
    GSEA_temp['gene'] = GSEA_temp.index
    
    # check to see if a folder exists to put the GSEA results
    if not os.path.exists(output_loc):
        os.mkdir(output_loc)
        
    if not os.path.exists(output_loc + 'rankfiles/'):
        os.mkdir(output_loc + 'rankfiles/')
    
    # drop all columns but score and gene name
    drop_labels = []
    for i in GSEA_temp.columns:
        if i != 'gene':
            if i != 'scores':
                drop_labels.append(i)
    GSEA_temp.drop(columns = drop_labels, inplace = True)
    
    # if filtering the genes
    if type(genes) != type(None):
        GSEA_temp = GSEA_temp[GSEA_temp['gene'].isin(genes)]
    
    #GSEA_temp.index = [i.upper() for i in GSEA_temp.gene]
    GSEA_temp.drop(columns = 'gene', inplace = True)
    GSEA_temp.to_csv(rankfile_loc, sep = '\t', header = False)

    scores, norm, pvals = decoupler.run_gsea(
        GSEA_temp.T,
        hallmark[hallmark["geneset"].isin(gsea_genesets)],
        source="geneset",
        target="genesymbol",
        min_n=5,)
    
    # save the results to a table
    gsea = (pd.concat({"enrichment_score": scores.T, "norm_enrichment": norm.T, "adj_pval": pvals.T}, axis=1).droplevel(level = 1, axis = 1).sort_values("enrichment_score", ascending = False))
    gsea.to_csv(output_loc + '{}.csv'.format(name))
    

# Load the data

In [ ]:
# Read in the data
lung_adata = sc.read_h5ad('/Genomics/chanlab/blaw/Spatial_Lung/raw_data/pipeline_obj/Lung_dbit_srf_CellReports_Science_with_pseudotime.h5ad')

In [ ]:
# Run using the top 3000 highly variable genes in the scRNA dataset
scRNA_only = lung_adata[(lung_adata.obs['tech'] == 'scRNA_seq')].copy()
highly_expressed = sc.pp.highly_variable_genes(scRNA_only, inplace = False, n_top_genes = 3000)
highly_expressed.index = scRNA_only.var.index
top_3000_genes = highly_expressed[highly_expressed['highly_variable'] == True].index

# Look at cluster marker genes
- To improve resolution, we will look at the scRNA only datasets

In [ ]:
scRNA_only = lung_adata[lung_adata.obs['tech'] == 'scRNA_seq'].copy()
sc.tl.rank_genes_groups(scRNA_only, 'cellstate', method='wilcoxon', use_raw = False, pts = True)
sc.pl.rank_genes_groups(scRNA_only, n_genes=30, sharey=False, use_raw = False)

In [ ]:
for i in scRNA_only.obs['cellstate'].unique():
    test_df = pd.DataFrame(index = scRNA_only.uns['rank_genes_groups']['names'][i])

    col_names = ['scores', 'logfoldchanges', 'pvals', 'pvals_adj', 'pts', 'pts_rest']

    for col in col_names:
        test_df[col] = scRNA_only.uns['rank_genes_groups'][col][i]
        
    genes = test_df.index
    test_df['{}_avg_expr'.format(i)] = scRNA_only[scRNA_only.obs['cellstate'] == i][:, genes].X.mean(axis = 0).tolist()[0]
    test_df['not_{}_avg_expr'.format(i)] = scRNA_only[scRNA_only.obs['cellstate'] != i][:, genes].X.mean(axis = 0).tolist()[0]
    test_df['manualLogFC'] = np.log2(test_df['{}_avg_expr'.format(i)] + 0.01) - np.log2(test_df['not_{}_avg_expr'.format(i)] + 0.01)
    
    test_df.to_csv('/Genomics/chanlab/blaw/Spatial_Lung/sandbox/scRNA/DE/marker_genes/cellstates/{}_vs_rest_scRNA_only.csv'.format(i))

In [ ]:
# Run GESEA for each marker gene analysis
for state in scRNA_only.obs['cellstate'].unique():
    runGSEApy(DE_file = '/Genomics/chanlab/blaw/Spatial_Lung/sandbox/scRNA/DE/marker_genes/cellstates/{}_vs_rest.csv'.format(state),
              gmt_file = '/Genomics/chanlab/blaw/Spatial_Lung/metadata/m5.all.v2023.1.Mm.symbols.gmt',
              output_folder = '/Genomics/chanlab/blaw/Spatial_Lung/sandbox/scRNA/DE/marker_genes/GSEA/',
              name = '{}_markers'.format(state))

### Plot a heatmap of marker genes

In [ ]:
# Calculate the avg expression of each cellstate
avg_expr_df = pd.DataFrame(columns = lung_adata.var.index, index = mes_states)

for i in avg_expr_df.index:
    avg_expr_df.loc[i] = lung_adata[lung_adata.obs['cellstate'] == i].X.mean(axis = 0).tolist()[0]
avg_expr_df.to_csv('/Genomics/chanlab/blaw/Spatial_Lung/sandbox/scRNA/DE/clusters/Mes_cluster_avg_expr.txt', sep = '\t')

In [ ]:
# calculate the zscores of genes
z_expr_df = avg_expr_df.copy()

# remove genes that have a std of 0 (this will break the z score)
bad_genes = []
for gene in z_expr_df.columns:
    if z_expr_df[gene].std() == 0:
        bad_genes.append(gene)  
z_expr_df.drop(columns = bad_genes, inplace = True)

# Calculate the Z score
mean_vals = z_expr_df.mean(axis = 0)
std_vals = z_expr_df.std(axis = 0)
z_expr_df = (z_expr_df - mean_vals) / (std_vals)
z_expr_df.fillna(0, inplace = True)

for col in z_expr_df.columns:
    z_expr_df[col] = z_expr_df[col].astype('float')

In [ ]:
# Select the set of the top 25 genes per cluster
top_marker_genes = []

for state in mes_states:
    #if state == 'SmoothMuscle':
     #   top_marker_genes.append('Myocd')
    #elif state == 'Precartilage':
     #   top_marker_genes.append('Sox9')
    temp_df = pd.read_csv('/Genomics/chanlab/blaw/Spatial_Lung/sandbox/scRNA/DE/marker_genes/cellstates/{}_vs_rest_scRNA_only.csv'.format(state), index_col = 0)
    for gene in temp_df[0:11].index:
        if gene not in top_marker_genes and gene in top_genes:
            top_marker_genes.append(gene)

### Plot volcano plots

In [ ]:
# Look at the MP marker genes
temp_df = pd.read_csv('/Genomics/chanlab/blaw/Spatial_Lung/sandbox/scRNA/DE/marker_genes/cellstates/MesenchymeProgenitor_vs_rest_scRNA_only.csv', index_col = 0)

# add pseudolowest pvals for any that are too low for the computer
temp_df['pvals_adj_2'] = temp_df['pvals_adj']
for gene in temp_df[temp_df['pvals_adj'] == 0].index:
    temp_df.loc[gene, 'pvals_adj_2'] = temp_df[temp_df['pvals_adj'] > 0]['pvals_adj'].min()

annotate_genes = ['Wnt2', 'Hoxb8', 'Top2a', 'Cdk1', 'Cenpa']

fig, ax = plt.subplots(figsize = (6, 5))
# Plot the volcano plot of the differential expression using manual LogFC
x = temp_df[(temp_df['MesenchymeProgenitor_avg_expr'] != 0) & (temp_df['not_MesenchymeProgenitor_avg_expr'] != 0)]['manualLogFC'].values
y = -np.log10(temp_df[(temp_df['MesenchymeProgenitor_avg_expr'] != 0) & (temp_df['not_MesenchymeProgenitor_avg_expr'] != 0)]['pvals_adj_2']).values
plt.plot(x, y, '.', alpha = 0.3, color = 'lightgray')

# highlight genes that are higher than 1.5 logfoldchange and higher than 1.3 -log10 adjustedpval
x2 = []
y2 = []
count = 0
for i in x:
    try:
        if x[count] > 1 and y[count] > 1.3:
            x2.append(x[count])
            y2.append(y[count])
        if x[count] < -1 and y[count] > 1.3:
            x2.append(x[count])
            y2.append(y[count])
    except:
        pass
    count += 1
plt.plot(x2, y2, '.')

for gene in annotate_genes:
    x = temp_df.loc[gene, 'manualLogFC']
    y = -np.log10(temp_df.loc[gene, 'pvals_adj_2'])
    
    plt.text(x + 0.1, y + 0.1, gene) 


plt.xlabel('LogFC')
plt.ylabel('-log10(adj_pval)')
plt.title('MP Marker Genes')
#plt.savefig('/Genomics/chanlab/blaw/Spatial_Lung/sandbox/scRNA/panels/Fig3/Fig.3h_SyntheticMesenchyme_markers_volcano_plot.pdf', dpi = 300)
plt.show()
plt.close()

In [ ]:
# Look at the SmoothMuscleProgenitor marker genes
temp_df = pd.read_csv('/Genomics/chanlab/blaw/Spatial_Lung/sandbox/scRNA/DE/marker_genes/cellstates/SmoothMuscleProgenitor_vs_rest_scRNA_only.csv', index_col = 0)

# add pseudolowest pvals for any that are too low for the computer
temp_df['pvals_adj_2'] = temp_df['pvals_adj']
for gene in temp_df[temp_df['pvals_adj'] == 0].index:
    temp_df.loc[gene, 'pvals_adj_2'] = temp_df[temp_df['pvals_adj'] > 0]['pvals_adj'].min()

annotate_genes = ['Cald1', 'Robo2', 'Ptch1', 'Acta2', 'Actg2', 'Ptch2', 'Myocd']

fig, ax = plt.subplots(figsize = (6, 5))
# Plot the volcano plot of the differential expression using manual LogFC
x = temp_df[(temp_df['SmoothMuscleProgenitor_avg_expr'] != 0) & (temp_df['not_SmoothMuscleProgenitor_avg_expr'] != 0)]['manualLogFC'].values
y = -np.log10(temp_df[(temp_df['SmoothMuscleProgenitor_avg_expr'] != 0) & (temp_df['not_SmoothMuscleProgenitor_avg_expr'] != 0)]['pvals_adj_2']).values
plt.plot(x, y, '.', alpha = 0.3, color = 'lightgray')

# highlight genes that are higher than 1.5 logfoldchange and higher than 1.3 -log10 adjustedpval
x2 = []
y2 = []
count = 0
for i in x:
    try:
        if x[count] > 1 and y[count] > 1.3:
            x2.append(x[count])
            y2.append(y[count])
        if x[count] < -1 and y[count] > 1.3:
            x2.append(x[count])
            y2.append(y[count])
    except:
        pass
    count += 1
plt.plot(x2, y2, '.')

for gene in annotate_genes:
    x = temp_df.loc[gene, 'manualLogFC']
    y = -np.log10(temp_df.loc[gene, 'pvals_adj_2'])
    
    plt.text(x + 0.1, y + 0.1, gene) 


plt.xlabel('LogFC')
plt.ylabel('-log10(adj_pval)')
plt.title('SmoothMuscleProgenitor Marker Genes')
#plt.savefig('/Genomics/chanlab/blaw/Spatial_Lung/sandbox/scRNA/panels/Fig3/Fig.3h_SyntheticMesenchyme_markers_volcano_plot.pdf', dpi = 300)
plt.show()
plt.close()

In [ ]:
# Look at the MatrixMesenchyme marker genes
temp_df = pd.read_csv('/Genomics/chanlab/blaw/Spatial_Lung/sandbox/scRNA/DE/marker_genes/cellstates/MatrixMesenchyme_vs_rest_scRNA_only.csv', index_col = 0)

# add pseudolowest pvals for any that are too low for the computer
temp_df['pvals_adj_2'] = temp_df['pvals_adj']
for gene in temp_df[temp_df['pvals_adj'] == 0].index:
    temp_df.loc[gene, 'pvals_adj_2'] = temp_df[temp_df['pvals_adj'] > 0]['pvals_adj'].min()

annotate_genes = ['Ptn', 'Prrx1', 'Col1a1', 'Osr1', 'Col1a2', 'Tmtc2', 'Col3a1', 'Mfap4', 'Igfbp5', 'Col5a2', 'Dcn', 'Adamtsl3']

fig, ax = plt.subplots(figsize = (10, 5))
# Plot the volcano plot of the differential expression using manual LogFC
x = temp_df[(temp_df['MatrixMesenchyme_avg_expr'] != 0) & (temp_df['not_MatrixMesenchyme_avg_expr'] != 0)]['manualLogFC'].values
y = -np.log10(temp_df[(temp_df['MatrixMesenchyme_avg_expr'] != 0) & (temp_df['not_MatrixMesenchyme_avg_expr'] != 0)]['pvals_adj_2']).values
plt.plot(x, y, '.', alpha = 0.3, color = 'lightgray')

# highlight genes that are higher than 1.5 logfoldchange and higher than 1.3 -log10 adjustedpval
x2 = []
y2 = []
count = 0
for i in x:
    try:
        if x[count] > 1.5 and y[count] > 1.3:
            x2.append(x[count])
            y2.append(y[count])
        #if x[count] < -1.5 and y[count] > 1.3:
         #   x2.append(x[count])
          #  y2.append(y[count])
    except:
        pass
    count += 1
plt.plot(x2, y2, '.', alpha = 0.5)

for gene in annotate_genes:
    x = temp_df.loc[gene, 'manualLogFC']
    y = -np.log10(temp_df.loc[gene, 'pvals_adj_2'])
    
    plt.text(x + 0.1, y + 0.1, gene) 


plt.xlabel('LogFC')
plt.ylabel('-log10(adj_pval)')
plt.title('Matrix Mesenchyme Markers')
plt.tight_layout()
plt.savefig('/Genomics/chanlab/blaw/Spatial_Lung/sandbox/scRNA/panels/Fig3/Fig.3h_MatrixMesenchyme_markers_volcano_plot.pdf', dpi = 300)
plt.show()
plt.close()

# DE Between Clusters
- For most comparisons, we ran the differential expression between just the scRNA replicates or just the DBiT replicates to reduce noise

### DE between Epithelial states

In [ ]:
# Instead of doing all the comparisons, I will be running DE and GSEA on all genes or only the top 3000 genes
# This will be run on the scRNA only cells
epithelial_comparisons = [('Epithelial(Distal)', 'Epithelial(Proximal)')]
output_dir = '/Genomics/chanlab/blaw/Spatial_Lung/sandbox/scRNA/DE/clusters/'

for comparison in epithelial_comparisons:    
    temp_adata = scRNA_only[scRNA_only.obs['cellstate'].isin(comparison)].copy()

    if len(temp_adata[temp_adata.obs['cellstate'] == comparison[0]]) > 0 and len(temp_adata[temp_adata.obs['cellstate'] == comparison[1]]) > 0:
        calculateDE(temp_adata,
                    'cellstate',
                    comparison,
                    output_dir,
                    'Lung')
        
        # run GSEA using all genes
        runGSEApy(DE_file = output_dir + 'gene_lists/{}_{}_{}_wilcox_DE.csv'.format('Lung', comparison[0], comparison[1]),
              gmt_file = '/Genomics/chanlab/blaw/Spatial_Lung/metadata/m5.all.v2023.1.Mm.symbols.gmt',
              output_folder = output_dir + 'GSEA/',
              name = '{}_{}'.format(comparison[0], comparison[1]),
              genes = None)
        
        # run GSEA using the top 3000 genes
        runGSEApy(DE_file = output_dir + 'gene_lists/{}_{}_{}_wilcox_DE.csv'.format('Lung', comparison[0], comparison[1]),
              gmt_file = '/Genomics/chanlab/blaw/Spatial_Lung/metadata/m5.all.v2023.1.Mm.symbols.gmt',
              output_folder = output_dir + 'GSEA_top3000/',
              name = '{}_{}'.format(comparison[0], comparison[1]),
              genes = top_genes)

### DE between Mesenchyme states

In [ ]:
# Instead of doing all the comparisons, I will be comparing the cell states within trajectories
meso_comparisons = list(itertools.combinations(['MesenchymeProgenitor', 'MatrixMesenchyme', 'Precartilage', 'SmoothMuscleProgenitor', 'SmoothMuscle'], 2))
output_dir = '/Genomics/chanlab/blaw/Spatial_Lung/sandbox/scRNA/DE/clusters/'

for comparison in meso_comparisons:    
    temp_adata = scRNA_only[scRNA_only.obs['cellstate'].isin(comparison)].copy()

    if len(temp_adata[temp_adata.obs['cellstate'] == comparison[0]]) > 0 and len(temp_adata[temp_adata.obs['cellstate'] == comparison[1]]) > 0:
        calculateDE(temp_adata,
                    'cellstate',
                    comparison,
                    output_dir,
                    'Lung')

        # run GSEA using all genes
        runGSEApy(DE_file = output_dir + 'gene_lists/{}_{}_{}_wilcox_DE.csv'.format('Lung', comparison[0], comparison[1]),
              gmt_file = '/Genomics/chanlab/blaw/Spatial_Lung/metadata/m5.all.v2023.1.Mm.symbols.gmt',
              output_folder = output_dir + 'GSEA/',
              name = '{}_{}'.format(comparison[0], comparison[1]),
              genes = None)
        
        # run GSEA using the top 3000 genes
        runGSEApy(DE_file = output_dir + 'gene_lists/{}_{}_{}_wilcox_DE.csv'.format('Lung', comparison[0], comparison[1]),
              gmt_file = '/Genomics/chanlab/blaw/Spatial_Lung/metadata/m5.all.v2023.1.Mm.symbols.gmt',
              output_folder = output_dir + 'GSEA_top3000/',
              name = '{}_{}'.format(comparison[0], comparison[1]),
              genes = top_genes)

### Plot volcano plots of the pairwise comparisons

In [ ]:
# Look at the synthetic mesenchyme vs precartilage
temp_df = pd.read_csv('/Genomics/chanlab/blaw/Spatial_Lung/sandbox/scRNA/DE/clusters/gene_lists/Lung_MatrixMesenchyme_Precartilage_wilcox_DE.csv', index_col = 0)

# add pseudolowest pvals for any that are too low for the computer
temp_df['pvals_adj_2'] = temp_df['pvals_adj']
for gene in temp_df[temp_df['pvals_adj'] == 0].index:
    temp_df.loc[gene, 'pvals_adj_2'] = temp_df[temp_df['pvals_adj'] > 0]['pvals_adj'].min()

annotate_genes = ['Ptn', 'Prrx1', 'Col1a1', 'Tmtc2', 'Sox9', 'Prdm16', 'Rspo2', 'Fgf10']

fig, ax = plt.subplots(figsize = (4, 5))
# Plot the volcano plot of the differential expression using manual LogFC
x = temp_df[(temp_df['MatrixMesenchyme_avg_expr'] != 0) & (temp_df['Precartilage_avg_expr'] != 0)]['manual_LogFC'].values
y = -np.log10(temp_df[(temp_df['MatrixMesenchyme_avg_expr'] != 0) & (temp_df['Precartilage_avg_expr'] != 0)]['pvals_adj_2']).values
plt.plot(x, y, '.', alpha = 0.3, color = 'lightgray')

# highlight genes that are higher than 1.5 logfoldchange and higher than 1.3 -log10 adjustedpval
x2 = []
y2 = []
count = 0
for i in x:
    try:
        if x[count] > 1 and y[count] > 1.3:
            x2.append(x[count])
            y2.append(y[count])
        if x[count] < -1 and y[count] > 1.3:
            x2.append(x[count])
            y2.append(y[count])
    except:
        pass
    count += 1
plt.plot(x2, y2, '.')

for gene in annotate_genes:
    x = temp_df.loc[gene, 'manual_LogFC']
    y = -np.log10(temp_df.loc[gene, 'pvals_adj_2'])
    
    plt.text(x + 0.1, y + 0.1, gene) 


plt.xlabel('LogFC')
plt.ylabel('-log10(adj_pval)')
plt.title('Matrix Mesenchyme vs Precartilage - Differential Expression')
#plt.savefig('/Genomics/chanlab/blaw/Spatial_Lung/sandbox/scRNA/panels/Fig3/Fig.3h_SyntheticMesenchyme_markers_volcano_plot.pdf', dpi = 300)
plt.show()
plt.close()

In [ ]:
# Look at the MesenchymeProgenitor vs synthetic mesenchyme
temp_df = pd.read_csv('/Genomics/chanlab/blaw/Spatial_Lung/sandbox/scRNA/DE/clusters/gene_lists/Lung_MesenchymeProgenitor_MatrixMesenchyme_wilcox_DE.csv', index_col = 0)

# add pseudolowest pvals for any that are too low for the computer
temp_df['pvals_adj_2'] = temp_df['pvals_adj']
for gene in temp_df[temp_df['pvals_adj'] == 0].index:
    temp_df.loc[gene, 'pvals_adj_2'] = temp_df[temp_df['pvals_adj'] > 0]['pvals_adj'].min()

annotate_genes = ['Ptn', 'Prrx1', 'Col1a1', 'Tmtc2', 'Sox9', 'Prdm16', 'Rspo2', 'Fgf10']

fig, ax = plt.subplots(figsize = (4, 5))
# Plot the volcano plot of the differential expression using manual LogFC
x = temp_df[(temp_df['MatrixMesenchyme_avg_expr'] != 0) & (temp_df['MesenchymeProgenitor_avg_expr'] != 0)]['manual_LogFC'].values
y = -np.log10(temp_df[(temp_df['MatrixMesenchyme_avg_expr'] != 0) & (temp_df['MesenchymeProgenitor_avg_expr'] != 0)]['pvals_adj_2']).values
plt.plot(x, y, '.', alpha = 0.3, color = 'lightgray')

# highlight genes that are higher than 1.5 logfoldchange and higher than 1.3 -log10 adjustedpval
x2 = []
y2 = []
count = 0
for i in x:
    try:
        if x[count] > 1 and y[count] > 1.3:
            x2.append(x[count])
            y2.append(y[count])
        if x[count] < -1 and y[count] > 1.3:
            x2.append(x[count])
            y2.append(y[count])
    except:
        pass
    count += 1
plt.plot(x2, y2, '.')

for gene in annotate_genes:
    x = temp_df.loc[gene, 'manual_LogFC']
    y = -np.log10(temp_df.loc[gene, 'pvals_adj_2'])
    
    plt.text(x + 0.1, y + 0.1, gene) 


plt.xlabel('LogFC')
plt.ylabel('-log10(adj_pval)')
plt.title('MesenchymeProgenitor vs MatrixMesenchyme - Differential Expression')
#plt.savefig('/Genomics/chanlab/blaw/Spatial_Lung/sandbox/scRNA/panels/Fig3/Fig.3h_SyntheticMesenchyme_markers_volcano_plot.pdf', dpi = 300)
plt.show()
plt.close()

In [ ]:
# Look at the MP vs SmoothMuscleProgenitor
temp_df = pd.read_csv('/Genomics/chanlab/blaw/Spatial_Lung/sandbox/scRNA/DE/clusters/gene_lists/Lung_MesenchymeProgenitor_SmoothMuscleProgenitor_wilcox_DE.csv', index_col = 0)

# add pseudolowest pvals for any that are too low for the computer
temp_df['pvals_adj_2'] = temp_df['pvals_adj']
for gene in temp_df[temp_df['pvals_adj'] == 0].index:
    temp_df.loc[gene, 'pvals_adj_2'] = temp_df[temp_df['pvals_adj'] > 0]['pvals_adj'].min()

annotate_genes = ['Ptn', 'Bmp4', 'Wnt2', 'Myocd', 'Ptch1', 'Ptch2', 'Actg2', 'Tgfbi', 'Cdk1', 'Cenpa', 'Top2a', 'Igf1', 'Cald1', 'Hoxb8', 'Asz1', 'Calb2']

fig, ax = plt.subplots(figsize = (8, 5))
# Plot the volcano plot of the differential expression using manual LogFC
x = temp_df[(temp_df['MesenchymeProgenitor_avg_expr'] != 0) & (temp_df['SmoothMuscleProgenitor_avg_expr'] != 0)]['manual_LogFC'].values
y = -np.log10(temp_df[(temp_df['MesenchymeProgenitor_avg_expr'] != 0) & (temp_df['SmoothMuscleProgenitor_avg_expr'] != 0)]['pvals_adj_2']).values
plt.plot(x, y, '.', alpha = 0.3, color = 'lightgray')

# highlight genes that are higher than 1.5 logfoldchange and higher than 1.3 -log10 adjustedpval
x2 = []
y2 = []
count = 0
for i in x:
    try:
        if x[count] > 1 and y[count] > 1:
            x2.append(x[count])
            y2.append(y[count])
        if x[count] < -1 and y[count] > 1:
            x2.append(x[count])
            y2.append(y[count])
    except:
        pass
    count += 1
plt.plot(x2, y2, '.')

for gene in annotate_genes:
    x = temp_df.loc[gene, 'manual_LogFC']
    y = -np.log10(temp_df.loc[gene, 'pvals_adj_2'])
    
    plt.text(x + 0.1, y + 0.1, gene) 


plt.xlabel('LogFC')
plt.ylabel('-log10(adj_pval)')
plt.title('MP vs SMP - Differential Expression')
#plt.savefig('/Genomics/chanlab/blaw/Spatial_Lung/sandbox/scRNA/panels/Fig3/Fig.3h_SyntheticMesenchyme_markers_volcano_plot.pdf', dpi = 300)
plt.show()
plt.close()

In [ ]:
# Look at the SmoothMuscleProgenitor vs SmoothMuscle
temp_df = pd.read_csv('/Genomics/chanlab/blaw/Spatial_Lung/sandbox/scRNA/DE/clusters/gene_lists/Lung_SmoothMuscleProgenitor_SmoothMuscle_wilcox_DE.csv', index_col = 0)

# add pseudolowest pvals for any that are too low for the computer
temp_df['pvals_adj_2'] = temp_df['pvals_adj']
for gene in temp_df[temp_df['pvals_adj'] == 0].index:
    temp_df.loc[gene, 'pvals_adj_2'] = temp_df[temp_df['pvals_adj'] > 0]['pvals_adj'].min()

annotate_genes = ['Myocd', 'Mylk', 'Acta2', 'Actg2', 'Rspo2', 'Fgf10', 'Snai2', 'Rps27a', 'Rpl32', 'Cdk1', 'Cenpa']

fig, ax = plt.subplots(figsize = (4, 5))
# Plot the volcano plot of the differential expression using manual LogFC
x = temp_df[(temp_df['SmoothMuscleProgenitor_avg_expr'] != 0) & (temp_df['SmoothMuscle_avg_expr'] != 0)]['manual_LogFC'].values
y = -np.log10(temp_df[(temp_df['SmoothMuscleProgenitor_avg_expr'] != 0) & (temp_df['SmoothMuscle_avg_expr'] != 0)]['pvals_adj_2']).values
plt.plot(x, y, '.', alpha = 0.3, color = 'lightgray')

# highlight genes that are higher than 1.5 logfoldchange and higher than 1.3 -log10 adjustedpval
x2 = []
y2 = []
count = 0
for i in x:
    try:
        if x[count] > 1 and y[count] > 1.3:
            x2.append(x[count])
            y2.append(y[count])
        if x[count] < -1 and y[count] > 1.3:
            x2.append(x[count])
            y2.append(y[count])
    except:
        pass
    count += 1
plt.plot(x2, y2, '.')

for gene in annotate_genes:
    x = temp_df.loc[gene, 'manual_LogFC']
    y = -np.log10(temp_df.loc[gene, 'pvals_adj_2'])
    
    plt.text(x + 0.1, y + 0.1, gene) 


plt.xlabel('LogFC')
plt.ylabel('-log10(adj_pval)')
plt.title('SmoothMuscleProgenitor vs SmoothMuscle - Differential Expression')
#plt.savefig('/Genomics/chanlab/blaw/Spatial_Lung/sandbox/scRNA/panels/Fig3/Fig.3h_SyntheticMesenchyme_markers_volcano_plot.pdf', dpi = 300)
plt.show()
plt.close()

### Heatmap of top genes

In [ ]:
expr_df = pd.DataFrame(columns = lung_adata.var.index, index = ['SmoothMuscle', 'SmoothMuscleProgenitor', 'MesenchymeProgenitor', 'Precartilage', 'SyntheticMesenchyme'])

for i in expr_df.index:
    #expr_df.loc[i] = epi_adata[(epi_adata.obs['subcluster_cellstate'] == i) & (epi_adata.obs['tech'] == 'scRNA_seq')].X.mean(axis = 0)
    expr_df.loc[i] = lung_adata[(lung_adata.obs['cellstate'] == i)].X.mean(axis = 0)
    
# calculate the zscores of genes
z_expr_df = expr_df.copy()

# remove genes that have a std of 0 (this will break the z score)
bad_genes = []
for gene in z_expr_df.columns:
    if z_expr_df[gene].std() == 0:
        bad_genes.append(gene)  
z_expr_df.drop(columns = bad_genes, inplace = True)

mean_vals = z_expr_df.mean(axis = 0)
std_vals = z_expr_df.std(axis = 0)

z_expr_df = (z_expr_df - mean_vals) / (std_vals)
z_expr_df.fillna(0, inplace = True)

for col in z_expr_df.columns:
    z_expr_df[col] = z_expr_df[col].astype('float')

temp = z_expr_df.T.copy()
genes = ['Acta2', 'Actg2', 'Myocd', 'Bmp4', 'Ptch1', 'Cdk1', 'Top2a', 'Wnt2', 'Rspo2', 'Sox9', 'Prdm16', 'Prrx1', 'Ptn', 'Col1a1', 'Fgf10', 'Hoxb5',  'Hoxb8', 'Asz1', 'Calb2']
drop_col = []
temp = temp.loc[genes]
colormap = cm.RdYlBu_r

g = sns.clustermap(temp, figsize = (5, 6), row_cluster = False, col_cluster = False, cmap=colormap, vmin = -2, vmax=2)
plt.subplots_adjust(bottom = 0.3, right = 0.7)
#plt.savefig('/Genomics/chanlab/blaw/Spatial_Lung/sandbox/scRNA/panels/Fig3/Fig.3i_mesenchyme_DE_heatmap.pdf', dpi = 300)
plt.show()
plt.close()

In [ ]:
temp = avg_expr_df.T.copy()
temp = temp.astype('float')
genes = [ 'Wnt2', 'Rspo2', 'Fgf10', 'Hoxb5', 'Hoxb8']
drop_col = []
temp = temp.loc[genes]
colormap = cm.RdYlBu_r

g = sns.clustermap(temp, figsize = (5, 6), row_cluster = False, col_cluster = False, cmap=reds, vmin = 0, vmax=2)
plt.subplots_adjust(bottom = 0.3, right = 0.7)
#plt.savefig('/Genomics/chanlab/blaw/Spatial_Lung/sandbox/scRNA/panels/Fig3/Fig.3i_mesenchyme_DE_heatmap.pdf', dpi = 300)
plt.show()
plt.close()

# DE between epithelial subclusters
- Since the DBIT does not evenly fill all the clusters, i'll perform this analysis on only the scRNA cells

In [ ]:
epi_adata = sc.read_h5ad('/Genomics/chanlab/blaw/Spatial_Lung/raw_data/pipeline_obj/Lung_epithelial_subcluster_pseudotime.h5ad')

In [ ]:
new_labels = []

for i in epi_adata.obs.index:
    if epi_adata.obs.loc[i, 'subcluster_cellstate'] in ['DistalIntermediate', 'ProximalIntermediate']:
        new_labels.append('Intermediate')
    else:
        new_labels.append(epi_adata.obs.loc[i, 'subcluster_cellstate'] )

epi_adata.obs['subcluster_cellstate'] = new_labels

In [ ]:
epi_scRNA_adata = epi_adata[epi_adata.obs['tech'] == 'scRNA_seq'].copy()

In [ ]:
# Run using the top 2000 highly variable genes
highly_expressed = sc.pp.highly_variable_genes(epi_scRNA_adata, inplace = False, n_top_genes = 2000)
highly_expressed.index = epi_scRNA_adata.var.index
top_epi_genes = highly_expressed[highly_expressed['highly_variable'] == True].index

### Find marker genes per subcluster

In [ ]:
temp_adata = epi_adata[epi_adata.obs['tech'] == 'scRNA_seq'].copy()
sc.tl.rank_genes_groups(temp_adata, 'subcluster_cellstate', method='wilcoxon', use_raw = False, pts = True)
sc.pl.rank_genes_groups(temp_adata,  n_genes=30, sharey=False, use_raw = False)

In [ ]:
for i in epi_adata.obs['subcluster_cellstate'].unique():
    test_df = pd.DataFrame(index = temp_adata.uns['rank_genes_groups']['names'][i])

    col_names = ['scores', 'logfoldchanges', 'pvals', 'pvals_adj', 'pts', 'pts_rest']

    for col in col_names:
        test_df[col] = temp_adata.uns['rank_genes_groups'][col][i]
        
    genes = test_df.index
    test_df['{}_avg_expr'.format(i)] = epi_adata[epi_adata.obs['subcluster_cellstate'] == i][:, genes].X.mean(axis = 0).tolist()[0]
    test_df['not_{}_avg_expr'.format(i)] = epi_adata[epi_adata.obs['subcluster_cellstate'] != i][:, genes].X.mean(axis = 0).tolist()[0]
    test_df['manualLogFC'] = np.log2(test_df['{}_avg_expr'.format(i)] + 0.01) - np.log2(test_df['not_{}_avg_expr'.format(i)] + 0.01)
    
    test_df.to_csv('/Genomics/chanlab/blaw/Spatial_Lung/sandbox/scRNA/DE/epithelial_subclusters/marker_genes/{}_vs_rest.csv'.format(i))

In [ ]:
for i in epi_adata.obs['subcluster_cellstate'].unique():
    test_df = pd.DataFrame(index = temp_adata.uns['rank_genes_groups']['names'][i])

    col_names = ['scores', 'logfoldchanges', 'pvals', 'pvals_adj', 'pts', 'pts_rest']

    for col in col_names:
        test_df[col] = temp_adata.uns['rank_genes_groups'][col][i]
        
    genes = test_df.index
    test_df['{}_avg_expr'.format(i)] = epi_adata[epi_adata.obs['subcluster_cellstate'] == i][:, genes].X.mean(axis = 0).tolist()[0]
    test_df['not_{}_avg_expr'.format(i)] = epi_adata[epi_adata.obs['subcluster_cellstate'] != i][:, genes].X.mean(axis = 0).tolist()[0]
    test_df['manualLogFC'] = np.log2(test_df['{}_avg_expr'.format(i)] + 0.01) - np.log2(test_df['not_{}_avg_expr'.format(i)] + 0.01)
    
    test_df.to_csv('/Genomics/chanlab/blaw/Spatial_Lung/sandbox/scRNA/DE/epithelial_subclusters/marker_genes/{}_vs_rest.csv'.format(i))

In [ ]:
total_genes = []

for i in epi_adata.obs['subcluster_cellstate'].unique():
    temp_df = pd.read_csv('/Genomics/chanlab/blaw/Spatial_Lung/sandbox/scRNA/DE/epithelial_subclusters/marker_genes/{}_vs_rest.csv'.format(i), index_col = 0)
    top_20 = temp_df.index[0:15]
    
    for gene in top_20:
        if not gene in total_genes:
            total_genes.append(gene)

In [ ]:
expr_df = pd.DataFrame(columns = epi_adata.var.index, index = subcluster_cellstates)

for i in expr_df.index:
    expr_df.loc[i] = epi_adata[(epi_adata.obs['subcluster_cellstate'] == i) & (epi_adata.obs['tech'] == 'scRNA_seq')].X.mean(axis = 0)
    #expr_df.loc[i] = epi_adata[(epi_adata.obs['subcluster_cellstate'] == i)].X.mean(axis = 0)
    
# calculate the zscores of genes
z_expr_df = expr_df.copy()

# remove genes that have a std of 0 (this will break the z score)
bad_genes = []
for gene in z_expr_df.columns:
    if z_expr_df[gene].std() == 0:
        bad_genes.append(gene)  
z_expr_df.drop(columns = bad_genes, inplace = True)

mean_vals = z_expr_df.mean(axis = 0)
std_vals = z_expr_df.std(axis = 0)

z_expr_df = (z_expr_df - mean_vals) / (std_vals)
z_expr_df.fillna(0, inplace = True)

for col in z_expr_df.columns:
    z_expr_df[col] = z_expr_df[col].astype('float')

In [ ]:
temp = z_expr_df.copy()
genes = ['Sftpb', 'Adamts18', 'Id2', 'Bmp4', 'Sox9', 'Etv5', 'Smoc2', 'Edn3', 'Icam1', 'Tbx1', 'Tgfb3', 'Ckb', 'Cdkn1c', 'Klf9', 'Mxd4', 'Sox2', 'Adamtsl1'] 
drop_col = []
for gene in temp.columns:
    if gene not in genes:
        drop_col.append(gene)
temp.drop(columns = drop_col, inplace = True)
colormap = cm.RdYlBu_r

g = sns.clustermap(temp[genes], figsize = (5, 3), row_cluster = False, col_cluster = False, cmap=colormap, vmin = -2, vmax=2)
plt.subplots_adjust(bottom = 0.3, right = 0.7)
plt.savefig('/Genomics/chanlab/blaw/Spatial_Lung/sandbox/scRNA/panels/Fig2/Fig.2f_subcluster_DE_heatmap.pdf', dpi = 300)
plt.show()
plt.close()

In [ ]:
# Look at the MP1 marker genes
temp_df = pd.read_csv('/Genomics/chanlab/blaw/Spatial_Lung/sandbox/scRNA/DE/epithelial_subclusters_marker_genes/DistalIntermediate_vs_rest.csv', index_col = 0)

# add pseudolowest pvals for any that are too low for the computer
temp_df['pvals_adj_2'] = temp_df['pvals_adj']
for gene in temp_df[temp_df['pvals_adj'] == 0].index:
    temp_df.loc[gene, 'pvals_adj_2'] = temp_df[temp_df['pvals_adj'] > 0]['pvals_adj'].min()

annotate_genes = ['Smoc2', 'Icam1']

fig, ax = plt.subplots(figsize = (4, 5))
# Plot the volcano plot of the differential expression using manual LogFC
x = temp_df[(temp_df['DistalIntermediate_avg_expr'] != 0) & (temp_df['not_DistalIntermediate_avg_expr'] != 0)]['manualLogFC'].values
y = -np.log10(temp_df[(temp_df['DistalIntermediate_avg_expr'] != 0) & (temp_df['not_DistalIntermediate_avg_expr'] != 0)]['pvals_adj_2']).values
plt.plot(x, y, '.', alpha = 0.3, color = 'lightgray')

# highlight genes that are higher than 1.5 logfoldchange and higher than 1.3 -log10 adjustedpval
x2 = []
y2 = []
count = 0
for i in x:
    try:
        if x[count] > 1 and y[count] > 1.3:
            x2.append(x[count])
            y2.append(y[count])
        if x[count] < -1 and y[count] > 1.3:
            x2.append(x[count])
            y2.append(y[count])
    except:
        pass
    count += 1
plt.plot(x2, y2, '.')

for gene in annotate_genes:
    x = temp_df.loc[gene, 'manualLogFC']
    y = -np.log10(temp_df.loc[gene, 'pvals_adj_2'])
    
    plt.text(x + 0.1, y + 0.1, gene) 


plt.xlabel('LogFC')
plt.ylabel('-log10(adj_pval)')
plt.title('DistalIntermediate Markers')
#plt.savefig('/Genomics/chanlab/blaw/Spatial_Lung/sandbox/scRNA/panels/Fig3/Fig.3h_SyntheticMesenchyme_markers_volcano_plot.pdf', dpi = 300)
plt.show()
plt.close()

### DE between subclusters

In [ ]:
epithelial_comparisons = list(itertools.combinations(epi_scRNA_adata.obs['subcluster_cellstate'].unique(), 2))
output_dir = '/Genomics/chanlab/blaw/Spatial_Lung/sandbox/scRNA/DE/epithelial_subclusters/'

for comparison in epithelial_comparisons:    
    temp_adata = epi_scRNA_adata[epi_scRNA_adata.obs['subcluster_cellstate'].isin(comparison)].copy()

    if len(temp_adata[temp_adata.obs['subcluster_cellstate'] == comparison[0]]) > 0 and len(temp_adata[temp_adata.obs['subcluster_cellstate'] == comparison[1]]) > 0:
        calculateDE(temp_adata,
                    'subcluster_cellstate',
                    comparison,
                    output_dir,
                    'epithelial_subclusters_scRNA_only')

        # run GSEA using all genes
        runGSEApy(DE_file = output_dir + 'gene_lists/{}_{}_{}_wilcox_DE.csv'.format('epithelial_subclusters_scRNA_only', comparison[0], comparison[1]),
                  gmt_file = '/Genomics/chanlab/blaw/Spatial_Lung/metadata/m5.all.v2023.1.Mm.symbols.gmt',
                  output_folder = output_dir + 'GSEA/',
                  name = '{}_{}'.format(comparison[0], comparison[1]),
                  genes = None)
        
        # run GSEA using the top 3000 genes
        runGSEApy(DE_file = output_dir + 'gene_lists/{}_{}_{}_wilcox_DE.csv'.format('epithelial_subclusters_scRNA_only', comparison[0], comparison[1]),
                  gmt_file = '/Genomics/chanlab/blaw/Spatial_Lung/metadata/m5.all.v2023.1.Mm.symbols.gmt',
                  output_folder = output_dir + 'GSEA_top3000/',
                  name = '{}_{}'.format(comparison[0], comparison[1]),
                  genes = top_genes)

In [ ]:
# Compare timepoints for the proximal cluster
timepoint_comparisons = list(itertools.combinations(['E11.5', 'E12.5', 'E13.5'], 2))
output_dir = '/Genomics/chanlab/blaw/Spatial_Lung/sandbox/scRNA/DE/timepoints/'

for state in ['Proximal', 'DistalIntermediate', 'ProximalIntermediate1']:
    label = state
    
    for comparison in timepoint_comparisons:    
        # run DE and GSEA with only scRNA cells
        temp_adata = epi_adata[(epi_adata.obs['timepoint'].isin(comparison)) & (epi_adata.obs['subcluster_cellstate'].isin([state]))].copy()
        name = 'scRNA_{}'.format(label)
        if len(temp_adata[temp_adata.obs['timepoint'] == comparison[0]]) > 0 and len(temp_adata[temp_adata.obs['timepoint'] == comparison[1]]) > 0:
            calculateDE(temp_adata,
                        'timepoint',
                        comparison,
                        output_dir,
                        name)

### Plot volcano plots between subclusters

In [ ]:
# Look at the synthetic mesenchyme
temp_df = pd.read_csv('/Genomics/chanlab/blaw/Spatial_Lung/sandbox/scRNA/DE/epithelial_subclusters/gene_lists/epithelial_subclusters_scRNA_only_DistalIntermediate_ProximalIntermediate_wilcox_DE.csv', index_col = 0)

# add pseudolowest pvals for any that are too low for the computer
temp_df['pvals_adj_2'] = temp_df['pvals_adj']
for gene in temp_df[temp_df['pvals_adj'] == 0].index:
    temp_df.loc[gene, 'pvals_adj_2'] = temp_df[temp_df['pvals_adj'] > 0]['pvals_adj'].min()

annotate_genes = ['Etv5', 'Sftpc', 'Cldn10', 'Aldh1a7', 'Adamtsl1', 'Sox2', 'Tbx1', 'Sox9']

fig, ax = plt.subplots(figsize = (4, 5))
# Plot the volcano plot of the differential expression using manual LogFC
x = temp_df[(temp_df['DistalIntermediate_avg_expr'] != 0) & (temp_df['ProximalIntermediate_avg_expr'] != 0)]['manual_LogFC'].values
y = -np.log10(temp_df[(temp_df['DistalIntermediate_avg_expr'] != 0) & (temp_df['ProximalIntermediate_avg_expr'] != 0)]['pvals_adj_2']).values
plt.plot(x, y, '.', alpha = 0.3, color = 'lightgray')

# highlight genes that are higher than 1.5 logfoldchange and higher than 1.3 -log10 adjustedpval
x2 = []
y2 = []
count = 0
for i in x:
    try:
        if x[count] > 1 and y[count] > 1.3:
            x2.append(x[count])
            y2.append(y[count])
        if x[count] < -1 and y[count] > 1.3:
            x2.append(x[count])
            y2.append(y[count])
    except:
        pass
    count += 1
plt.plot(x2, y2, '.')

for gene in annotate_genes:
    x = temp_df.loc[gene, 'manual_LogFC']
    y = -np.log10(temp_df.loc[gene, 'pvals_adj_2'])
    
    plt.text(x + 0.1, y + 0.1, gene) 


plt.xlabel('LogFC')
plt.ylabel('-log10(adj_pval)')
plt.title('DI vs PI - Differential Expression')
plt.savefig('/Genomics/chanlab/blaw/Spatial_Lung/sandbox/scRNA/panels/Fig2/Fig.2j_DI_vs_PI_markers_volcano_plot.pdf', dpi = 300)
plt.show()
plt.close()

# DE between mesenchyme subclusters

In [ ]:
mes_adata = sc.read_h5ad('/Genomics/chanlab/blaw/Spatial_Lung/raw_data/pipeline_obj/Lung_mesenchyme_subcluster_pseudotime.h5ad')

In [ ]:
mes_scRNA_adata = mes_adata[mes_adata.obs['tech'] == 'scRNA_seq'].copy()

# Run using the top 2000 highly variable genes
highly_expressed = sc.pp.highly_variable_genes(mes_scRNA_adata, inplace = False, n_top_genes = 3000)
highly_expressed.index = mes_scRNA_adata.var.index
top_mes_genes = highly_expressed[highly_expressed['highly_variable'] == True].index

In [ ]:
temp_adata = mes_adata[mes_adata.obs['tech'] == 'scRNA_seq'].copy()
sc.tl.rank_genes_groups(temp_adata, 'subcluster_cellstate', method='wilcoxon', use_raw = False, pts = True)
sc.pl.rank_genes_groups(temp_adata,  n_genes=30, sharey=False, use_raw = False)

In [ ]:
genes = ['Wnt2', 'Sox9', 'Prdm16', 'Fgf10', 'Rspo2']
sc.pl.violin(mes_scRNA_adata,
             genes, groupby = 'subcluster_cellstate', jitter=0.4, multi_panel= True, rotation = 45)

In [ ]:
genes = ['Wnt2', 'Sox9', 'Prdm16', 'Fgf10', 'Rspo2']
sc.pl.violin(mes_adata[mes_adata.obs['tech'] == 'DBiT_seq'],
             genes, groupby = 'subcluster_cellstate', jitter=0.4, multi_panel= True, rotation = 90)

In [ ]:
for i in mes_adata.obs['subcluster_cellstate'].unique():
    test_df = pd.DataFrame(index = temp_adata.uns['rank_genes_groups']['names'][i])

    col_names = ['scores', 'logfoldchanges', 'pvals', 'pvals_adj', 'pts', 'pts_rest']

    for col in col_names:
        test_df[col] = temp_adata.uns['rank_genes_groups'][col][i]
        
    genes = test_df.index
    test_df['{}_avg_expr'.format(i)] = mes_adata[mes_adata.obs['subcluster_cellstate'] == i][:, genes].X.mean(axis = 0).tolist()[0]
    test_df['not_{}_avg_expr'.format(i)] = mes_adata[mes_adata.obs['subcluster_cellstate'] != i][:, genes].X.mean(axis = 0).tolist()[0]
    test_df['manualLogFC'] = np.log2(test_df['{}_avg_expr'.format(i)] + 0.01) - np.log2(test_df['not_{}_avg_expr'.format(i)] + 0.01)
    
    test_df.to_csv('/Genomics/chanlab/blaw/Spatial_Lung/sandbox/scRNA/DE/mesenchyme_subclusters/marker_genes/{}_vs_rest.csv'.format(i))

### Run DE between mesenchyme subclusters

In [ ]:
mes_comparisons = list(itertools.combinations(mes_scRNA_adata.obs['subcluster_cellstate'].unique(), 2))
output_dir = '/Genomics/chanlab/blaw/Spatial_Lung/sandbox/scRNA/DE/mesenchyme_subclusters/'

for comparison in mes_comparisons:    
    temp_adata = mes_scRNA_adata[mes_scRNA_adata.obs['subcluster_cellstate'].isin(comparison)].copy()

    if len(temp_adata[temp_adata.obs['subcluster_cellstate'] == comparison[0]]) > 0 and len(temp_adata[temp_adata.obs['subcluster_cellstate'] == comparison[1]]) > 0:
        calculateDE(temp_adata,
                    'subcluster_cellstate',
                    comparison,
                    output_dir,
                    'mesenchyme_subclusters_scRNA_only')

        # run GSEA using all genes
        runGSEApy(DE_file = output_dir + 'gene_lists/{}_{}_{}_wilcox_DE.csv'.format('mesenchyme_subclusters_scRNA_only', comparison[0], comparison[1]),
                  gmt_file = '/Genomics/chanlab/blaw/Spatial_Lung/metadata/m5.all.v2023.1.Mm.symbols.gmt',
                  output_folder = output_dir + 'GSEA/',
                  name = '{}_{}'.format(comparison[0], comparison[1]),
                  genes = None)
        
        # run GSEA using the top 3000 genes
        runGSEApy(DE_file = output_dir + 'gene_lists/{}_{}_{}_wilcox_DE.csv'.format('mesenchyme_subclusters_scRNA_only', comparison[0], comparison[1]),
                  gmt_file = '/Genomics/chanlab/blaw/Spatial_Lung/metadata/m5.all.v2023.1.Mm.symbols.gmt',
                  output_folder = output_dir + 'GSEA_top3000/',
                  name = '{}_{}'.format(comparison[0], comparison[1]),
                  genes = top_genes)

In [ ]:
# Look at the synthetic mesenchyme
temp_df = pd.read_csv('/Genomics/chanlab/blaw/Spatial_Lung/sandbox/scRNA/DE/mesenchyme_subclusters/gene_lists/mesenchyme_subclusters_scRNA_only_MP2_MP1_wilcox_DE.csv', index_col = 0)

# add pseudolowest pvals for any that are too low for the computer
temp_df['pvals_adj_2'] = temp_df['pvals_adj']
for gene in temp_df[temp_df['pvals_adj'] == 0].index:
    temp_df.loc[gene, 'pvals_adj_2'] = temp_df[temp_df['pvals_adj'] > 0]['pvals_adj'].min()

annotate_genes = ['Wnt2', 'Ptn', 'Col1a1', 'Rspo2', 'Fgf10', 'Cdk1', 'Cenpa', 'Ube2c', 'Top2a', 'Igfbp5', 'Dcn', 'Mfap4']

fig, ax = plt.subplots(figsize = (4, 5))
# Plot the volcano plot of the differential expression using manual LogFC
x = temp_df[(temp_df['MP1_avg_expr'] != 0) & (temp_df['MP2_avg_expr'] != 0)]['manual_LogFC'].values
y = -np.log10(temp_df[(temp_df['MP1_avg_expr'] != 0) & (temp_df['MP2_avg_expr'] != 0)]['pvals_adj_2']).values
plt.plot(x, y, '.', alpha = 0.3, color = 'lightgray')

# highlight genes that are higher than 1.5 logfoldchange and higher than 1.3 -log10 adjustedpval
x2 = []
y2 = []
count = 0
for i in x:
    try:
        if x[count] > 1 and y[count] > 1.3:
            x2.append(x[count])
            y2.append(y[count])
        if x[count] < -1 and y[count] > 1.3:
            x2.append(x[count])
            y2.append(y[count])
    except:
        pass
    count += 1
plt.plot(x2, y2, '.')

for gene in annotate_genes:
    x = temp_df.loc[gene, 'manual_LogFC']
    y = -np.log10(temp_df.loc[gene, 'pvals_adj_2'])
    
    plt.text(x + 0.1, y + 0.1, gene) 


plt.xlabel('LogFC')
plt.ylabel('-log10(adj_pval)')
plt.title('MP2 vs MP1 - Differential Expression')
#plt.savefig('/Genomics/chanlab/blaw/Spatial_Lung/sandbox/scRNA/panels/Fig2/Fig.2j_DI_vs_PI_markers_volcano_plot.pdf', dpi = 300)
plt.show()
plt.close()

In [ ]:
temp_df = pd.read_csv('/Genomics/chanlab/blaw/Spatial_Lung/sandbox/scRNA/DE/mesenchyme_subclusters/gene_lists/mesenchyme_subclusters_scRNA_only_MP2_SyntheticMesenchymeProgenitor_wilcox_DE.csv', index_col = 0)


In [ ]:
temp_df[-50:]

In [ ]:
# Plot and save a volcano plot
plot_volcano('/Genomics/chanlab/blaw/Spatial_Lung/sandbox/scRNA/DE/mesenchyme_subclusters/gene_lists/mesenchyme_subclusters_scRNA_only_MP2_SyntheticMesenchymeProgenitor_wilcox_DE.csv', 
             upName = 'MP2',
             downName = 'SyntheticMesenchymeProgenitor',
             annotate_genes = ['Mfap4', 'Prrx1', 'Osr1', 'Dcn', 'Col1a1', 'Ptn', 'Fgf10', 'Rspo2', 'Wnt2', 'Hoxb6', 'Snai2', 'Hoxb5', 'Prdm16', 'Sox9'],
             show = True,
             palette = mesSubclusterColors,
             size = (5, 5))

In [ ]:
# Look at the synthetic mesenchyme
temp_df = pd.read_csv('/Genomics/chanlab/blaw/Spatial_Lung/sandbox/scRNA/DE/mesenchyme_subclusters/gene_lists/mesenchyme_subclusters_scRNA_only_MP1_SmoothMuscleProgenitor_wilcox_DE.csv', index_col = 0)

# add pseudolowest pvals for any that are too low for the computer
temp_df['pvals_adj_2'] = temp_df['pvals_adj']
for gene in temp_df[temp_df['pvals_adj'] == 0].index:
    temp_df.loc[gene, 'pvals_adj_2'] = temp_df[temp_df['pvals_adj'] > 0]['pvals_adj'].min()

annotate_genes = ['Wnt2', 'Ptn', 'Hoxb8', 'Hoxb6', 'Hoxb5', 'Tgfbi', 'Cald1', 'Ptch2', 'Myocd', 'Acta2', 'Actg2']

fig, ax = plt.subplots(figsize = (4, 5))
# Plot the volcano plot of the differential expression using manual LogFC
x = temp_df[(temp_df['MP1_avg_expr'] != 0) & (temp_df['SmoothMuscleProgenitor_avg_expr'] != 0)]['manual_LogFC'].values
y = -np.log10(temp_df[(temp_df['MP1_avg_expr'] != 0) & (temp_df['SmoothMuscleProgenitor_avg_expr'] != 0)]['pvals_adj_2']).values
plt.plot(x, y, '.', alpha = 0.3, color = 'lightgray')

# highlight genes that are higher than 1.5 logfoldchange and higher than 1.3 -log10 adjustedpval
x2 = []
y2 = []
count = 0
for i in x:
    try:
        if x[count] > 1 and y[count] > 1.3:
            x2.append(x[count])
            y2.append(y[count])
        if x[count] < -1 and y[count] > 1.3:
            x2.append(x[count])
            y2.append(y[count])
    except:
        pass
    count += 1
plt.plot(x2, y2, '.')

for gene in annotate_genes:
    x = temp_df.loc[gene, 'manual_LogFC']
    y = -np.log10(temp_df.loc[gene, 'pvals_adj_2'])
    
    plt.text(x + 0.1, y + 0.1, gene) 


plt.xlabel('LogFC')
plt.ylabel('-log10(adj_pval)')
plt.title('MP1 vs SMP - Differential Expression')
#plt.savefig('/Genomics/chanlab/blaw/Spatial_Lung/sandbox/scRNA/panels/Fig2/Fig.2j_DI_vs_PI_markers_volcano_plot.pdf', dpi = 300)
plt.show()
plt.close()

In [ ]:
# Look at the synthetic mesenchyme
temp_df = pd.read_csv('/Genomics/chanlab/blaw/Spatial_Lung/sandbox/scRNA/DE/mesenchyme_subclusters/gene_lists/mesenchyme_subclusters_scRNA_only_Precartilage_SyntheticMesenchymeProgenitor_wilcox_DE.csv', index_col = 0)

# add pseudolowest pvals for any that are too low for the computer
temp_df['pvals_adj_2'] = temp_df['pvals_adj']
for gene in temp_df[temp_df['pvals_adj'] == 0].index:
    temp_df.loc[gene, 'pvals_adj_2'] = temp_df[temp_df['pvals_adj'] > 0]['pvals_adj'].min()

annotate_genes = ['Ptn', 'Prrx1', 'Mfap4', 'Prdm16','Osr1', 'Dcn', 'Col1a1', 'Ran', 'Cdk4', 'Cks2', 'Ptma']

fig, ax = plt.subplots(figsize = (4, 5))
# Plot the volcano plot of the differential expression using manual LogFC
x = temp_df[(temp_df['Precartilage_avg_expr'] != 0) & (temp_df['SyntheticMesenchymeProgenitor_avg_expr'] != 0)]['manual_LogFC'].values
y = -np.log10(temp_df[(temp_df['SyntheticMesenchyme_avg_expr'] != 0) & (temp_df['SyntheticMesenchymeProgenitor_avg_expr'] != 0)]['pvals_adj_2']).values
plt.plot(x, y, '.', alpha = 0.3, color = 'lightgray')

# highlight genes that are higher than 1.5 logfoldchange and higher than 1.3 -log10 adjustedpval
x2 = []
y2 = []
count = 0
for i in x:
    try:
        if x[count] > 0.8 and y[count] > 1.3:
            x2.append(x[count])
            y2.append(y[count])
        if x[count] < -0.8 and y[count] > 1.3:
            x2.append(x[count])
            y2.append(y[count])
    except:
        pass
    count += 1
plt.plot(x2, y2, '.')

for gene in annotate_genes:
    x = temp_df.loc[gene, 'manual_LogFC']
    y = -np.log10(temp_df.loc[gene, 'pvals_adj_2'])
    
    plt.text(x + 0.1, y + 0.1, gene) 


plt.xlabel('LogFC')
plt.ylabel('-log10(adj_pval)')
plt.title('SyntheticMesenchyme vs SyntheticMesenchymeProgenitor')
#plt.savefig('/Genomics/chanlab/blaw/Spatial_Lung/sandbox/scRNA/panels/Fig2/Fig.2j_DI_vs_PI_markers_volcano_plot.pdf', dpi = 300)
plt.show()
plt.close()

In [ ]:
genes = ['Prdm16', 'Sox9']
order = ['MP1', 'MP2', 'MP3', 'SyntheticMesenchymeProgenitor', 'SyntheticMesenchyme', 'Precartilage']
sc.pl.violin(mes_scRNA_adata[mes_scRNA_adata.obs['subcluster_cellstate'].isin(order)],            
             genes, order = order, groupby = 'subcluster_cellstate', jitter=0.4, multi_panel= True, rotation = 90)

# DE between Branching and non-Branching annotations

In [ ]:
states = ['MesenchymeProgenitor', 'Epithelial(Distal)', 'Mesothelium', 'VascularEndothelial']
comparison = ('Branching', 'Not_Branching')

for state in states:    
    output_dir = '/Genomics/chanlab/blaw/Spatial_Lung/sandbox/scRNA/DE/branching_annotation/{}/'.format(state)
    if not os.path.exists(output_dir):
        os.mkdir(output_dir)

    temp_adata = lung_adata[(lung_adata.obs['cellstate'] == state) & (lung_adata.obs['branching_annotation'].isin(['Branching', 'Not_Branching']))].copy()

    if len(temp_adata[temp_adata.obs['branching_annotation'] == 'Branching']) > 0 and len(temp_adata[temp_adata.obs['branching_annotation'] == 'Not_Branching']) > 0:
        calculateDE(temp_adata,
                    'branching_annotation',
                    comparison,
                    output_dir,
                    '{}_branching'.format(state))

        # run GSEA using all genes
        runGSEApy(DE_file = output_dir + 'gene_lists/{}_{}_{}_wilcox_DE.csv'.format('{}_branching'.format(state), comparison[0], comparison[1]),
                  gmt_file = '/Genomics/chanlab/blaw/Spatial_Lung/metadata/m5.all.v2023.1.Mm.symbols.gmt',
                  output_folder = output_dir + 'GSEA/',
                  name = '{}_{}'.format(comparison[0], comparison[1]),
                  genes = None)
        
        # run GSEA using the top 3000 genes
        runGSEApy(DE_file = output_dir + 'gene_lists/{}_{}_{}_wilcox_DE.csv'.format('{}_branching'.format(state), comparison[0], comparison[1]),
                  gmt_file = '/Genomics/chanlab/blaw/Spatial_Lung/metadata/m5.all.v2023.1.Mm.symbols.gmt',
                  output_folder = output_dir + 'GSEA_top3000/',
                  name = '{}_{}'.format(comparison[0], comparison[1]),
                  genes = top_3000_genes)

In [ ]:
states = ['MesenchymeProgenitor', 'Epithelial(Distal)', 'Mesothelium', 'VascularEndothelial']
comparison = ('E11.5', 'E12.5')

for state in states:    
    output_dir = '/Genomics/chanlab/blaw/Spatial_Lung/sandbox/scRNA/DE/branching_annotation_timepoint/{}/'.format(state)
    if not os.path.exists(output_dir):
        os.mkdir(output_dir)

    temp_adata = lung_adata[(lung_adata.obs['cellstate'] == state) & (lung_adata.obs['branching_annotation'] == 'Branching') & (lung_adata.obs['timepoint'].isin(comparison))].copy()
    print(state, len(temp_adata))

In [ ]:
states = ['MesenchymeProgenitor', 'Epithelial(Distal)', 'Mesothelium', 'VascularEndothelial']
comparison = ('E11.5', 'E12.5')

for state in states:    
    output_dir = '/Genomics/chanlab/blaw/Spatial_Lung/sandbox/scRNA/DE/branching_annotation_timepoint/{}/'.format(state)
    if not os.path.exists(output_dir):
        os.mkdir(output_dir)

    temp_adata = lung_adata[(lung_adata.obs['cellstate'] == state) & (lung_adata.obs['branching_annotation'] == 'Branching') & (lung_adata.obs['timepoint'].isin(comparison))].copy()

    if len(temp_adata[temp_adata.obs['timepoint'] == 'E11.5']) > 10 and len(temp_adata[temp_adata.obs['timepoint'] == 'E12.5']) > 0:
        print('Passed:', state)
        calculateDE(temp_adata,
                    'timepoint',
                    comparison,
                    output_dir,
                    '{}_branching'.format(state))

        # run GSEA using all genes
        runGSEApy(DE_file = output_dir + 'gene_lists/{}_{}_{}_wilcox_DE.csv'.format('{}_branching'.format(state), comparison[0], comparison[1]),
                  gmt_file = '/Genomics/chanlab/blaw/Spatial_Lung/metadata/m5.all.v2023.1.Mm.symbols.gmt',
                  output_folder = output_dir + 'GSEA/',
                  name = '{}_{}'.format(comparison[0], comparison[1]),
                  genes = None)
        
        # run GSEA using the top 3000 genes
        runGSEApy(DE_file = output_dir + 'gene_lists/{}_{}_{}_wilcox_DE.csv'.format('{}_branching'.format(state), comparison[0], comparison[1]),
                  gmt_file = '/Genomics/chanlab/blaw/Spatial_Lung/metadata/m5.all.v2023.1.Mm.symbols.gmt',
                  output_folder = output_dir + 'GSEA_top3000/',
                  name = '{}_{}'.format(comparison[0], comparison[1]),
                  genes = top_3000_genes)

In [ ]:
timepoint_comparisons = [('E11.5', 'E12.5'), ('E11.5', 'E13.5'), ('E12.5', 'E13.5')]
output_dir = '/Genomics/chanlab/blaw/Spatial_Lung/sandbox/scRNA/DE/timepoints/epi_subclusters/scRNA/'

for state in epi_subcluster_cellstates:
    for comparison in timepoint_comparisons:    
        label = state
        # run DE and GSEA with only scRNA cells
        temp_adata = epi_adata[(epi_adata.obs['tech'] == 'scRNA_seq') & (epi_adata.obs['timepoint'].isin(comparison)) & (epi_adata.obs['subcluster_cellstate'].isin([state]))].copy()
        name = '{}'.format(label)
        # only keep comparisons that have more than 50 cells per comparison
        if len(temp_adata[temp_adata.obs['timepoint'] == comparison[0]]) > 50 and len(temp_adata[temp_adata.obs['timepoint'] == comparison[1]]) > 50:
            calculateDE(temp_adata,
                        'timepoint',
                        comparison,
                        output_dir,
                        name)
            
            # run GSEA using all genes
            runGSEApy(DE_file = output_dir + 'gene_lists/{}_{}_{}_wilcox_DE.csv'.format(name, comparison[0], comparison[1]),
                  gmt_file = '/Genomics/chanlab/blaw/Spatial_Lung/metadata/m5.all.v2023.1.Mm.symbols.gmt',
                  output_folder = output_dir + 'GSEA/',
                  name = '{}_{}_{}'.format(name, comparison[0], comparison[1]),
                  genes = None)
        
            # run GSEA using the top 3000 genes
            runGSEApy(DE_file = output_dir + 'gene_lists/{}_{}_{}_wilcox_DE.csv'.format(name, comparison[0], comparison[1]),
                  gmt_file = '/Genomics/chanlab/blaw/Spatial_Lung/metadata/m5.all.v2023.1.Mm.symbols.gmt',
                  output_folder = output_dir + 'GSEA_top3000/',
                  name = '{}_{}_{}'.format(name, comparison[0], comparison[1]),
                  genes = top_3000_genes)

In [ ]:
timepoint_comparisons = [('E11.5', 'E12.5'), ('E11.5', 'E13.5'), ('E12.5', 'E13.5')]
output_dir = '/Genomics/chanlab/blaw/Spatial_Lung/sandbox/scRNA/DE/timepoints/mes_subclusters/scRNA/'

for state in mes_subcluster_cellstates:
    for comparison in timepoint_comparisons:    
        label = state
        # run DE and GSEA with only scRNA cells
        temp_adata = mes_adata[(mes_adata.obs['tech'] == 'scRNA_seq') & (mes_adata.obs['timepoint'].isin(comparison)) & (mes_adata.obs['subcluster_cellstate'].isin([state]))].copy()
        name = '{}'.format(label)
        # only keep comparisons that have more than 50 cells per comparison
        if len(temp_adata[temp_adata.obs['timepoint'] == comparison[0]]) > 50 and len(temp_adata[temp_adata.obs['timepoint'] == comparison[1]]) > 50:
            calculateDE(temp_adata,
                        'timepoint',
                        comparison,
                        output_dir,
                        name)
            
            # run GSEA using all genes
            runGSEApy(DE_file = output_dir + 'gene_lists/{}_{}_{}_wilcox_DE.csv'.format(name, comparison[0], comparison[1]),
                  gmt_file = '/Genomics/chanlab/blaw/Spatial_Lung/metadata/m5.all.v2023.1.Mm.symbols.gmt',
                  output_folder = output_dir + 'GSEA/',
                  name = '{}_{}_{}'.format(name, comparison[0], comparison[1]),
                  genes = None)
        
            # run GSEA using the top 3000 genes
            runGSEApy(DE_file = output_dir + 'gene_lists/{}_{}_{}_wilcox_DE.csv'.format(name, comparison[0], comparison[1]),
                  gmt_file = '/Genomics/chanlab/blaw/Spatial_Lung/metadata/m5.all.v2023.1.Mm.symbols.gmt',
                  output_folder = output_dir + 'GSEA_top3000/',
                  name = '{}_{}_{}'.format(name, comparison[0], comparison[1]),
                  genes = top_3000_genes)

In [ ]:
timepoint_comparisons = [('E11.5', 'E12.5'), ('E11.5', 'E13.5'), ('E12.5', 'E13.5')]
output_dir = '/Genomics/chanlab/blaw/Spatial_Lung/sandbox/scRNA/DE/timepoints/clusters/DBiT/'

for state in cellstates:
    # change the name of epithelial(distal) and epithelial(proximal) to '_' to fix file names
    if state == 'Epithelial(Distal)':
        label = 'Epithelial_Distal'
    elif state == 'Epithelial(Proximal)':
        label = 'Epithelial_Proximal'
    else:
        label = state
        
    for comparison in timepoint_comparisons:    
        # run DE and GSEA with only scRNA cells
        temp_adata = lung_adata[(lung_adata.obs['tech'] == 'DBiT_seq') & (lung_adata.obs['timepoint'].isin(comparison)) & (lung_adata.obs['cellstate'].isin([state]))].copy()
        name = '{}'.format(label)
        # only keep comparisons that have more than 50 cells per comparison
        if len(temp_adata[temp_adata.obs['timepoint'] == comparison[0]]) > 50 and len(temp_adata[temp_adata.obs['timepoint'] == comparison[1]]) > 50:
            calculateDE(temp_adata,
                        'timepoint',
                        comparison,
                        output_dir,
                        name)
            
            # run GSEA using all genes
            runGSEApy(DE_file = output_dir + 'gene_lists/{}_{}_{}_wilcox_DE.csv'.format(name, comparison[0], comparison[1]),
                  gmt_file = '/Genomics/chanlab/blaw/Spatial_Lung/metadata/m5.all.v2023.1.Mm.symbols.gmt',
                  output_folder = output_dir + 'GSEA/',
                  name = '{}_{}_{}'.format(name, comparison[0], comparison[1]),
                  genes = None)
        
            # run GSEA using the top 3000 genes
            runGSEApy(DE_file = output_dir + 'gene_lists/{}_{}_{}_wilcox_DE.csv'.format(name, comparison[0], comparison[1]),
                  gmt_file = '/Genomics/chanlab/blaw/Spatial_Lung/metadata/m5.all.v2023.1.Mm.symbols.gmt',
                  output_folder = output_dir + 'GSEA_top3000/',
                  name = '{}_{}_{}'.format(name, comparison[0], comparison[1]),
                  genes = top_3000_genes)

In [ ]:
timepoint_comparisons = [('E11.5', 'E12.5'), ('E11.5', 'E13.5'), ('E12.5', 'E13.5')]
output_dir = '/Genomics/chanlab/blaw/Spatial_Lung/sandbox/scRNA/DE/timepoints/epi_subclusters/DBiT/'

for state in epi_subcluster_cellstates:
    for comparison in timepoint_comparisons:    
        label = state
        # run DE and GSEA with only DBIT cells
        temp_adata = epi_adata[(epi_adata.obs['tech'] == 'DBiT_seq') & (epi_adata.obs['timepoint'].isin(comparison)) & (epi_adata.obs['subcluster_cellstate'].isin([state]))].copy()
        name = '{}'.format(label)
        # only keep comparisons that have more than 50 cells per comparison
        if len(temp_adata[temp_adata.obs['timepoint'] == comparison[0]]) > 50 and len(temp_adata[temp_adata.obs['timepoint'] == comparison[1]]) > 50:
            calculateDE(temp_adata,
                        'timepoint',
                        comparison,
                        output_dir,
                        name)
            
            # run GSEA using all genes
            runGSEApy(DE_file = output_dir + 'gene_lists/{}_{}_{}_wilcox_DE.csv'.format(name, comparison[0], comparison[1]),
                  gmt_file = '/Genomics/chanlab/blaw/Spatial_Lung/metadata/m5.all.v2023.1.Mm.symbols.gmt',
                  output_folder = output_dir + 'GSEA/',
                  name = '{}_{}_{}'.format(name, comparison[0], comparison[1]),
                  genes = None)
        
            # run GSEA using the top 3000 genes
            runGSEApy(DE_file = output_dir + 'gene_lists/{}_{}_{}_wilcox_DE.csv'.format(name, comparison[0], comparison[1]),
                  gmt_file = '/Genomics/chanlab/blaw/Spatial_Lung/metadata/m5.all.v2023.1.Mm.symbols.gmt',
                  output_folder = output_dir + 'GSEA_top3000/',
                  name = '{}_{}_{}'.format(name, comparison[0], comparison[1]),
                  genes = top_3000_genes)

In [ ]:
timepoint_comparisons = [('E11.5', 'E12.5'), ('E11.5', 'E13.5'), ('E12.5', 'E13.5')]
output_dir = '/Genomics/chanlab/blaw/Spatial_Lung/sandbox/scRNA/DE/timepoints/mes_subclusters/DBiT/'

for state in mes_subcluster_cellstates:
    for comparison in timepoint_comparisons:    
        label = state
        # run DE and GSEA with only scRNA cells
        temp_adata = mes_adata[(mes_adata.obs['tech'] == 'DBiT_seq') & (mes_adata.obs['timepoint'].isin(comparison)) & (mes_adata.obs['subcluster_cellstate'].isin([state]))].copy()
        name = '{}'.format(label)
        # only keep comparisons that have more than 50 cells per comparison
        if len(temp_adata[temp_adata.obs['timepoint'] == comparison[0]]) > 50 and len(temp_adata[temp_adata.obs['timepoint'] == comparison[1]]) > 50:
            calculateDE(temp_adata,
                        'timepoint',
                        comparison,
                        output_dir,
                        name)
            
            # run GSEA using all genes
            runGSEApy(DE_file = output_dir + 'gene_lists/{}_{}_{}_wilcox_DE.csv'.format(name, comparison[0], comparison[1]),
                  gmt_file = '/Genomics/chanlab/blaw/Spatial_Lung/metadata/m5.all.v2023.1.Mm.symbols.gmt',
                  output_folder = output_dir + 'GSEA/',
                  name = '{}_{}_{}'.format(name, comparison[0], comparison[1]),
                  genes = None)
        
            # run GSEA using the top 3000 genes
            runGSEApy(DE_file = output_dir + 'gene_lists/{}_{}_{}_wilcox_DE.csv'.format(name, comparison[0], comparison[1]),
                  gmt_file = '/Genomics/chanlab/blaw/Spatial_Lung/metadata/m5.all.v2023.1.Mm.symbols.gmt',
                  output_folder = output_dir + 'GSEA_top3000/',
                  name = '{}_{}_{}'.format(name, comparison[0], comparison[1]),
                  genes = top_3000_genes)

In [ ]:
lung_adata.obs['Acta2_expr'] = lung_adata[:,'Acta2'].X.todense()
lung_adata.obs['Actg2_expr'] = lung_adata[:,'Actg2'].X.todense()
lung_adata.obs['Pdgfrb_expr'] = lung_adata[:,'Pdgfrb'].X.todense()
lung_adata.obs['Ptch1_expr'] = lung_adata[:,'Ptch1'].X.todense()

In [ ]:
fig, ax = plt.subplots(figsize = (10, 10))

cells = lung_adata.obs[(lung_adata.obs['cellstate'].isin(['MesenchymeProgenitor'])) & (lung_adata.obs['Acta2_expr'] != 0) & (lung_adata.obs['Ptch1_expr'] != 0)].index
x = lung_adata.obs.loc[cells, 'Acta2_expr']
y = lung_adata.obs.loc[cells, 'Ptch1_expr']

heatmap, xedges, yedges = np.histogram2d(x, y, bins=50)
extent = [xedges[0], xedges[-1], yedges[0], yedges[-1]]

plt.clf()
plt.imshow(heatmap.T, extent=extent, origin='lower')
plt.xlabel('acta2 expr')
plt.ylabel('Ptch1 expr')
plt.show()
plt.close()

In [ ]:
fig, ax = plt.subplots(figsize = (10, 10))

for cellstate in cellstates:
    cells = lung_adata.obs[lung_adata.obs['cellstate'] == cellstate].index
    prrx1_expr = lung_adata[cells, 'Prrx1'].X.mean()
    sox9_expr = lung_adata[cells, 'Sox9'].X.mean()
    
    plt.plot(prrx1_expr, sox9_expr, '.', color = clusterColors[cellstate])
    
plt.legend(cellstates, bbox_to_anchor = (1.01, 1.01))
plt.xlabel('Mean Prrx1_expr')
plt.ylabel('Mean Sox9_expr')
plt.show()
plt.close()

In [ ]:
lung_adata.obs['Prrx1_expr'] = lung_adata[:,'Prrx1'].X.todense()
lung_adata.obs['Sox9_expr'] = lung_adata[:,'Sox9'].X.todense()
lung_adata.obs['Osr1_expr'] = lung_adata[:,'Osr1'].X.todense()

In [ ]:
lung_adata.obs['Prrx1_Sox9'] = ''

for cell in lung_adata.obs.index:
    if lung_adata.obs.loc[cell, 'Prrx1_expr'] > 1 and lung_adata.obs.loc[cell, 'Sox9_expr'] > 1:
        lung_adata.obs.loc[cell, 'Prrx1_Sox9'] = 'Prrx1High/Sox9High'
    elif lung_adata.obs.loc[cell, 'Prrx1_expr'] > 1:
        lung_adata.obs.loc[cell, 'Prrx1_Sox9'] = 'Prrx1High/Sox9Low'
    elif lung_adata.obs.loc[cell, 'Sox9_expr'] > 1:
        lung_adata.obs.loc[cell, 'Prrx1_Sox9'] = 'Prrx1Low/Sox9High'
    else:
        lung_adata.obs.loc[cell, 'Prrx1_Sox9'] = 'Prrx1Low/Sox9Low'

In [ ]:
lung_adata.obs['Osr1_Sox9'] = ''

for cell in lung_adata.obs.index:
    if lung_adata.obs.loc[cell, 'Osr1_expr'] > 1 and lung_adata.obs.loc[cell, 'Sox9_expr'] > 1:
        lung_adata.obs.loc[cell, 'Osr1_Sox9'] = 'Osr1High/Sox9High'
    elif lung_adata.obs.loc[cell, 'Osr1_expr'] > 1:
        lung_adata.obs.loc[cell, 'Osr1_Sox9'] = 'Osr1High/Sox9Low'
    elif lung_adata.obs.loc[cell, 'Sox9_expr'] > 1:
        lung_adata.obs.loc[cell, 'Osr1_Sox9'] = 'Osr1Low/Sox9High'
    else:
        lung_adata.obs.loc[cell, 'Osr1_Sox9'] = 'Osr1Low/Sox9Low'

In [ ]:
prrx1_sox9Colors = {'Prrx1High/Sox9High': 'Blue', 'Prrx1High/Sox9Low': 'Red', 'Prrx1Low/Sox9High': 'Green', 'Prrx1Low/Sox9Low': 'lightgray'}
osr1_sox9Colors = {'Osr1High/Sox9High': 'Blue', 'Osr1High/Sox9Low': 'Red', 'Osr1Low/Sox9High': 'Green', 'Osr1Low/Sox9Low': 'lightgray'}

In [ ]:
sns.kdeplot(lung_adata.obs[lung_adata.obs['cellstate'].isin(['Epithelial(Distal)', 'SyntheticMesenchyme'])]['Prrx1_expr'])
sns.kdeplot(lung_adata.obs[lung_adata.obs['cellstate'].isin(['Epithelial(Distal)', 'SyntheticMesenchyme'])]['Sox9_expr'])
sns.kdeplot(lung_adata.obs[lung_adata.obs['cellstate'].isin(['Epithelial(Distal)', 'SyntheticMesenchyme'])]['Osr1_expr'])
plt.legend(['Prrx1', 'Sox9', 'Osr1'])
plt.title('Epithelial and SyntheticMesenchyme Cells')
plt.xlim(0)
plt.show()
plt.close()

In [ ]:
with rc_context({'figure.figsize': (5, 5)}):
    sc.pl.umap(lung_adata, color = ['Prrx1_Sox9'],size = 5, wspace = 0.35, palette = prrx1_sox9Colors)

In [ ]:
with rc_context({'figure.figsize': (5, 5)}):
    sc.pl.umap(lung_adata, color = ['Osr1_Sox9'],size = 5, wspace = 0.35, palette = osr1_sox9Colors)

In [ ]:
plot_spatial_groups(lung_adata,
                    'Prrx1_Sox9',
                    palette = prrx1_sox9Colors)

# Look at the presence of Sox2 / Sox9 double positive epithelial cells

In [ ]:
lung_adata.obs['Sox2_expr'] = lung_adata[:,'Sox2'].X.todense()
lung_adata.obs['Sox9_expr'] = lung_adata[:,'Sox9'].X.todense()

lung_adata.obs['Sox2_Sox9'] = ''

for cell in lung_adata.obs.index:
    if lung_adata.obs.loc[cell, 'Sox2_expr'] > 1 and lung_adata.obs.loc[cell, 'Sox9_expr'] > 1:
        lung_adata.obs.loc[cell, 'Sox2_Sox9'] = 'Sox2High/Sox9High'
    elif lung_adata.obs.loc[cell, 'Sox2_expr'] > 1:
        lung_adata.obs.loc[cell, 'Sox2_Sox9'] = 'Sox2High/Sox9Low'
    elif lung_adata.obs.loc[cell, 'Sox2_expr'] > 1:
        lung_adata.obs.loc[cell, 'Sox2_Sox9'] = 'Sox2Low/Sox9High'
    else:
        lung_adata.obs.loc[cell, 'Sox2_Sox9'] = 'Sox2Low/Sox9Low'

In [ ]:
double_positive = pd.DataFrame(index = ['E11.5', 'E12.5', 'E13.5'], columns = lung_adata.obs['Sox2_Sox9'].unique())

for timepoint in timepoints:
    for t in lung_adata.obs['Sox2_Sox9'].unique():
        double_positive.loc[timepoint, t] = len(lung_adata.obs[(lung_adata.obs['tech'] == 'DBiT_seq') & (lung_adata.obs['timepoint'] == timepoint) & (lung_adata.obs['Sox2_Sox9'] == t)  & (lung_adata.obs['cellstate'].isin(['Epithelial(Distal)', 'Epithelial(Proximal)']))])

In [ ]:
double_positive_norm = double_positive.copy()

for i in double_positive_norm.index:
    double_positive_norm.loc[i] = double_positive_norm.loc[i] / double_positive_norm.loc[i].sum()

fig, ax = plt.subplots(figsize = (4, 5))
double_positive_norm.plot(kind='bar', use_index = True, stacked=True, grid = False, ax = ax)
plt.legend(loc='center left', bbox_to_anchor=(1.0, 0.5))
plt.title('Sox2/Sox9')
plt.ylabel('% cells')
plt.tight_layout()
#plt.savefig('/Genomics/chanlab/blaw/Spatial_Lung/sandbox/scRNA/panels/Fig.1d_composition_per_timepoint_DBiT_only.pdf', dpi = 300)
plt.show()
plt.close()

In [ ]:
double_positive_norm = double_positive.copy()

for i in double_positive_norm.index:
    double_positive_norm.loc[i] = double_positive_norm.loc[i] / double_positive_norm.loc[i].sum()

fig, ax = plt.subplots(figsize = (4, 5))
double_positive_norm.plot(kind='bar', use_index = True, stacked=True, grid = False, ax = ax)
plt.legend(loc='center left', bbox_to_anchor=(1.0, 0.5))
plt.title('Sox2/Sox9 - DBiT only')
plt.ylabel('% cells')
plt.tight_layout()
#plt.savefig('/Genomics/chanlab/blaw/Spatial_Lung/sandbox/scRNA/panels/Fig.1d_composition_per_timepoint_DBiT_only.pdf', dpi = 300)
plt.show()
plt.close()

In [ ]:
# Instead of doing all the comparisons, I will be comparing the cell states within trajectories
timepoint_comparisons = list(itertools.combinations(['E11.5', 'E12.5', 'E13.5'], 2))
output_dir = '/Genomics/chanlab/blaw/Spatial_Lung/sandbox/scRNA/DE/timepoints/'

for state in ['Epithelial(Distal)', 'Epithelial(Proximal)']:
    if state == 'Epithelial(Distal)':
        label = 'Epithelial_Distal'
    elif state == 'Epithelial(Proximal)':
        label = 'Epithelial_Proximal'
    
    for comparison in timepoint_comparisons:    
        # run DE and GSEA with all cells
        temp_adata = lung_adata[(lung_adata.obs['timepoint'].isin(comparison)) & (lung_adata.obs['cellstate'].isin([state]))].copy()
        name = 'All_{}'.format(label)
        if len(temp_adata[temp_adata.obs['timepoint'] == comparison[0]]) > 0 and len(temp_adata[temp_adata.obs['timepoint'] == comparison[1]]) > 0:
            calculateDE(temp_adata,
                        'timepoint',
                        comparison,
                        output_dir,
                        name)

            # run decoupler using all genes
            runDecoupler(DE_file = output_dir + 'gene_lists/{}_{}_{}_wilcox_DE.csv'.format(name, comparison[0], comparison[1]),
                         gmt_file = "/Genomics/chanlab/blaw/Spatial_Lung/metadata/m5.all.v2023.1.Mm.symbols.gmt",
                         output_loc = output_dir + 'GSEA/',
                         name = '{}_{}_{}'.format(name, comparison[0], comparison[1]),
                         genes = None,
                         gene_set_min = 15,
                         gene_set_max = 500)

            # run decoupler using all genes
            runDecoupler(DE_file = output_dir + 'gene_lists/{}_{}_{}_wilcox_DE.csv'.format(name, comparison[0], comparison[1]),
                         gmt_file = "/Genomics/chanlab/blaw/Spatial_Lung/metadata/m5.all.v2023.1.Mm.symbols.gmt",
                         output_loc = output_dir + 'GSEA_top3000/',
                         name = '{}_{}_{}'.format(name, comparison[0], comparison[1]),
                         genes = top_genes,
                         gene_set_min = 15,
                         gene_set_max = 500)
            
        # run DE and GSEA with only DBIT cells
        temp_adata = lung_adata[(lung_adata.obs['tech'] == 'DBiT_seq') & (lung_adata.obs['timepoint'].isin(comparison)) & (lung_adata.obs['cellstate'].isin([state]))].copy()
        name = 'DBiT_{}'.format(label)
        if len(temp_adata[temp_adata.obs['timepoint'] == comparison[0]]) > 0 and len(temp_adata[temp_adata.obs['timepoint'] == comparison[1]]) > 0:
            calculateDE(temp_adata,
                        'timepoint',
                        comparison,
                        output_dir,
                        name)

            # run decoupler using all genes
            runDecoupler(DE_file = output_dir + 'gene_lists/{}_{}_{}_wilcox_DE.csv'.format(name, comparison[0], comparison[1]),
                         gmt_file = "/Genomics/chanlab/blaw/Spatial_Lung/metadata/m5.all.v2023.1.Mm.symbols.gmt",
                         output_loc = output_dir + 'GSEA/',
                         name = '{}_{}_{}'.format(name, comparison[0], comparison[1]),
                         genes = None,
                         gene_set_min = 15,
                         gene_set_max = 500)

            # run decoupler using all genes
            runDecoupler(DE_file = output_dir + 'gene_lists/{}_{}_{}_wilcox_DE.csv'.format(name, comparison[0], comparison[1]),
                         gmt_file = "/Genomics/chanlab/blaw/Spatial_Lung/metadata/m5.all.v2023.1.Mm.symbols.gmt",
                         output_loc = output_dir + 'GSEA_top3000/',
                         name = '{}_{}_{}'.format(name, comparison[0], comparison[1]),
                         genes = top_genes,
                         gene_set_min = 15,
                         gene_set_max = 500)

In [ ]:
# look at cleaned vs non-cleaned mes1 cells
mes1 = lung_adata[lung_adata.obs['cellstate'] == 'Mesenchyme1'].copy()
output_dir = '/Genomics/chanlab/blaw/Spatial_Lung/sandbox/scRNA/DE/mesenchyme1/'

comparison = ('clean', 'None')
temp_adata =  mes1[mes1.obs['clean'].isin(comparison)].copy()

if len(temp_adata[temp_adata.obs['clean'] == comparison[0]]) > 0 and len(temp_adata[temp_adata.obs['clean'] == comparison[1]]) > 0:
    calculateDE(temp_adata,
                'clean',
                comparison,
                output_dir,
                'Lung_mes1')

In [ ]:
# look at trachea vs lobe in mes1 cells
mes1 = lung_adata[lung_adata.obs['cellstate'] == 'Mesenchyme1'].copy()
output_dir = '/Genomics/chanlab/blaw/Spatial_Lung/sandbox/scRNA/DE/mesenchyme1/'

comparison = ('trachea', 'lobe')
temp_adata =  mes1[mes1.obs['lobe_trachea'].isin(comparison)].copy()

if len(temp_adata[temp_adata.obs['lobe_trachea'] == comparison[0]]) > 0 and len(temp_adata[temp_adata.obs['lobe_trachea'] == comparison[1]]) > 0:
    calculateDE(temp_adata,
                'lobe_trachea',
                comparison,
                output_dir,
                'Lung_mes1')

In [ ]:
# Instead of doing all the comparisons, I will be comparing the cell states within trajectories
timepoint_comparisons = list(itertools.combinations(['E12.5', 'E13.5'], 2))
output_dir = '/Genomics/chanlab/blaw/Spatial_Lung/sandbox/scRNA/DE/timepoints/'

for state in cellstates:
    # change the name of epithelial(distal) and epithelial(proximal) to '_' to save file names
    if state == 'Epithelial(Distal)':
        label = 'Epithelial_Distal'
    elif state == 'Epithelial(Proximal)':
        label = 'Epithelial_Proximal'
    else:
        label = state
        
    for comparison in timepoint_comparisons:    
        # run DE and GSEA with only scRNA cells
        temp_adata = lung_adata[(lung_adata.obs['tech'] == 'scRNA_seq') & (lung_adata.obs['timepoint'].isin(comparison)) & (lung_adata.obs['cellstate'].isin([state]))].copy()
        name = '{}'.format(label)
        # only keep comparisons that have more than 50 cells per comparison
        if len(temp_adata[temp_adata.obs['timepoint'] == comparison[0]]) > 50 and len(temp_adata[temp_adata.obs['timepoint'] == comparison[1]]) > 50:
            calculateDE(temp_adata,
                        'timepoint',
                        comparison,
                        output_dir,
                        name)
            
            # run GSEA using all genes
            runGSEApy(DE_file = output_dir + 'gene_lists/{}_{}_{}_wilcox_DE.csv'.format(name, comparison[0], comparison[1]),
                  gmt_file = '/Genomics/chanlab/blaw/Spatial_Lung/metadata/m5.all.v2023.1.Mm.symbols.gmt',
                  output_folder = output_dir + 'GSEA/',
                  name = '{}_{}_{}'.format(name, comparison[0], comparison[1]),
                  genes = None)
        
            # run GSEA using the top 3000 genes
            runGSEApy(DE_file = output_dir + 'gene_lists/{}_{}_{}_wilcox_DE.csv'.format(name, comparison[0], comparison[1]),
                  gmt_file = '/Genomics/chanlab/blaw/Spatial_Lung/metadata/m5.all.v2023.1.Mm.symbols.gmt',
                  output_folder = output_dir + 'GSEA_top3000/',
                  name = '{}_{}_{}'.format(name, comparison[0], comparison[1]),
                  genes = top_3000_genes)

2024-09-16 20:01:09,360 [INFO] Start to generate gseapy reports, and produce figures...
2024-09-16 20:01:09,361 [INFO] Congratulations. GSEApy runs successfully................

2024-09-16 20:01:14,971 [WARNING] Duplicated values found in preranked stats: 43.41% of genes
The order of those genes will be arbitrary, which may produce unexpected results.
2024-09-16 20:01:14,973 [INFO] Parsing data files for GSEA.............................
2024-09-16 20:01:15,353 [INFO] 5332 gene_sets have been filtered out when max_size=500 and min_size=15
2024-09-16 20:01:15,355 [INFO] 5433 gene_sets used for further statistical testing.....
2024-09-16 20:01:15,356 [INFO] Start to run GSEA...Might take a while..................
2024-09-16 20:09:52,785 [INFO] Start to generate gseapy reports, and produce figures...
2024-09-16 20:09:52,786 [INFO] Congratulations. GSEApy runs successfully................

2024-09-16 20:09:55,179 [WARNING] Duplicated values found in preranked stats: 68.77% of genes
The ord

2024-09-16 20:45:22,669 [INFO] Start to run GSEA...Might take a while..................
2024-09-16 20:53:48,541 [INFO] Start to generate gseapy reports, and produce figures...
2024-09-16 20:53:48,542 [INFO] Congratulations. GSEApy runs successfully................

2024-09-16 20:53:50,849 [WARNING] Duplicated values found in preranked stats: 77.20% of genes
The order of those genes will be arbitrary, which may produce unexpected results.
2024-09-16 20:53:50,850 [INFO] Parsing data files for GSEA.............................
2024-09-16 20:53:51,166 [INFO] 9488 gene_sets have been filtered out when max_size=500 and min_size=15
2024-09-16 20:53:51,167 [INFO] 1277 gene_sets used for further statistical testing.....
2024-09-16 20:53:51,168 [INFO] Start to run GSEA...Might take a while..................
2024-09-16 20:53:58,367 [INFO] Start to generate gseapy reports, and produce figures...
2024-09-16 20:53:58,368 [INFO] Congratulations. GSEApy runs successfully................

2024-09-16 20

2024-09-16 22:04:39,179 [INFO] 9488 gene_sets have been filtered out when max_size=500 and min_size=15
2024-09-16 22:04:39,181 [INFO] 1277 gene_sets used for further statistical testing.....
2024-09-16 22:04:39,182 [INFO] Start to run GSEA...Might take a while..................
2024-09-16 22:04:58,724 [INFO] Start to generate gseapy reports, and produce figures...
2024-09-16 22:04:58,726 [INFO] Congratulations. GSEApy runs successfully................

2024-09-16 22:05:01,261 [WARNING] Duplicated values found in preranked stats: 85.50% of genes
The order of those genes will be arbitrary, which may produce unexpected results.
2024-09-16 22:05:01,262 [INFO] Parsing data files for GSEA.............................
2024-09-16 22:05:01,642 [INFO] 5332 gene_sets have been filtered out when max_size=500 and min_size=15
2024-09-16 22:05:01,644 [INFO] 5433 gene_sets used for further statistical testing.....
2024-09-16 22:05:01,644 [INFO] Start to run GSEA...Might take a while..................


2024-09-16 22:58:38,735 [INFO] Start to generate gseapy reports, and produce figures...
2024-09-16 22:58:38,736 [INFO] Congratulations. GSEApy runs successfully................

2024-09-16 22:58:41,067 [WARNING] Duplicated values found in preranked stats: 51.47% of genes
The order of those genes will be arbitrary, which may produce unexpected results.
2024-09-16 22:58:41,067 [INFO] Parsing data files for GSEA.............................
2024-09-16 22:58:41,385 [INFO] 9488 gene_sets have been filtered out when max_size=500 and min_size=15
2024-09-16 22:58:41,386 [INFO] 1277 gene_sets used for further statistical testing.....
2024-09-16 22:58:41,387 [INFO] Start to run GSEA...Might take a while..................
2024-09-16 22:59:00,760 [INFO] Start to generate gseapy reports, and produce figures...
2024-09-16 22:59:00,762 [INFO] Congratulations. GSEApy runs successfully................

2024-09-16 22:59:03,119 [WARNING] Duplicated values found in preranked stats: 75.15% of genes
The ord

2024-09-16 23:42:17,421 [INFO] Parsing data files for GSEA.............................
2024-09-16 23:42:17,797 [INFO] 5332 gene_sets have been filtered out when max_size=500 and min_size=15
2024-09-16 23:42:17,799 [INFO] 5433 gene_sets used for further statistical testing.....
2024-09-16 23:42:17,800 [INFO] Start to run GSEA...Might take a while..................
2024-09-16 23:50:52,083 [INFO] Start to generate gseapy reports, and produce figures...
2024-09-16 23:50:52,084 [INFO] Congratulations. GSEApy runs successfully................

2024-09-16 23:50:54,398 [WARNING] Duplicated values found in preranked stats: 39.47% of genes
The order of those genes will be arbitrary, which may produce unexpected results.
2024-09-16 23:50:54,399 [INFO] Parsing data files for GSEA.............................
2024-09-16 23:50:54,732 [INFO] 9488 gene_sets have been filtered out when max_size=500 and min_size=15
2024-09-16 23:50:54,733 [INFO] 1277 gene_sets used for further statistical testing.....


2024-09-17 00:22:59,077 [INFO] Start to run GSEA...Might take a while..................
2024-09-17 00:23:15,495 [INFO] Start to generate gseapy reports, and produce figures...
2024-09-17 00:23:15,496 [INFO] Congratulations. GSEApy runs successfully................

2024-09-17 00:23:18,426 [WARNING] Duplicated values found in preranked stats: 65.26% of genes
The order of those genes will be arbitrary, which may produce unexpected results.
2024-09-17 00:23:18,427 [INFO] Parsing data files for GSEA.............................
2024-09-17 00:23:18,812 [INFO] 5332 gene_sets have been filtered out when max_size=500 and min_size=15
2024-09-17 00:23:18,814 [INFO] 5433 gene_sets used for further statistical testing.....
2024-09-17 00:23:18,815 [INFO] Start to run GSEA...Might take a while..................
2024-09-17 00:27:24,571 [INFO] Start to generate gseapy reports, and produce figures...
2024-09-17 00:27:24,572 [INFO] Congratulations. GSEApy runs successfully................

2024-09-17 00

2024-09-17 00:50:47,800 [WARNING] Duplicated values found in preranked stats: 52.93% of genes
The order of those genes will be arbitrary, which may produce unexpected results.
2024-09-17 00:50:47,800 [INFO] Parsing data files for GSEA.............................
2024-09-17 00:50:48,128 [INFO] 9488 gene_sets have been filtered out when max_size=500 and min_size=15
2024-09-17 00:50:48,130 [INFO] 1277 gene_sets used for further statistical testing.....
2024-09-17 00:50:48,130 [INFO] Start to run GSEA...Might take a while..................
2024-09-17 00:51:03,348 [INFO] Start to generate gseapy reports, and produce figures...
2024-09-17 00:51:03,349 [INFO] Congratulations. GSEApy runs successfully................

2024-09-17 00:51:05,201 [WARNING] Duplicated values found in preranked stats: 84.80% of genes
The order of those genes will be arbitrary, which may produce unexpected results.
2024-09-17 00:51:05,202 [INFO] Parsing data files for GSEA.............................
2024-09-17 00:5

2024-09-17 01:14:33,524 [INFO] 5332 gene_sets have been filtered out when max_size=500 and min_size=15
2024-09-17 01:14:33,526 [INFO] 5433 gene_sets used for further statistical testing.....
2024-09-17 01:14:33,527 [INFO] Start to run GSEA...Might take a while..................
2024-09-17 01:18:53,604 [INFO] Start to generate gseapy reports, and produce figures...
2024-09-17 01:18:53,605 [INFO] Congratulations. GSEApy runs successfully................

2024-09-17 01:18:55,901 [WARNING] Duplicated values found in preranked stats: 50.00% of genes
The order of those genes will be arbitrary, which may produce unexpected results.
2024-09-17 01:18:55,901 [INFO] Parsing data files for GSEA.............................
2024-09-17 01:18:56,242 [INFO] 9488 gene_sets have been filtered out when max_size=500 and min_size=15
2024-09-17 01:18:56,245 [INFO] 1277 gene_sets used for further statistical testing.....
2024-09-17 01:18:56,245 [INFO] Start to run GSEA...Might take a while..................


2024-09-17 01:42:29,720 [INFO] Start to generate gseapy reports, and produce figures...
2024-09-17 01:42:29,721 [INFO] Congratulations. GSEApy runs successfully................



In [ ]:
good_states = []
for state in cellstates:
    if len(lung_adata.obs[(lung_adata.obs['tech'] == 'scRNA_seq') & (lung_adata.obs['cellstate'] == state)]) >= 100:
        if len(lung_adata.obs[(lung_adata.obs['tech'] == 'DBiT_seq') & (lung_adata.obs['cellstate'] == state)]) >= 100:
            good_states.append(state)

In [ ]:
tests = []
for state in good_states:
    tests.append('{}_scRNA'.format(state))
    tests.append('{}_DBiT'.format(state))
gene_expr = pd.DataFrame(index = lung_adata.var_names, columns = tests)
raw_gene_counts = pd.DataFrame(index = lung_adata.var_names, columns = tests)

In [ ]:
for state in cellstates:
    raw_gene_counts['{}_scRNA'.format(state)] = lung_adata[(lung_adata.obs['tech'] == 'scRNA_seq') & (lung_adata.obs['cellstate'] == state)].layers['Raw_Counts'].mean(axis = 0).tolist()[0]
    raw_gene_counts['{}_DBiT'.format(state)] = lung_adata[(lung_adata.obs['tech'] == 'DBiT_seq') & (lung_adata.obs['cellstate'] == state)].layers['Raw_Counts'].mean(axis = 0).tolist()[0]

for state in cellstates:
    gene_expr['{}_scRNA'.format(state)] = lung_adata[(lung_adata.obs['tech'] == 'scRNA_seq') & (lung_adata.obs['cellstate'] == state)].X.mean(axis = 0).tolist()[0]
    gene_expr['{}_DBiT'.format(state)] = lung_adata[(lung_adata.obs['tech'] == 'DBiT_seq') & (lung_adata.obs['cellstate'] == state)].X.mean(axis = 0).tolist()[0]

gene_expr.to_csv('/Genomics/chanlab/blaw/Spatial_Lung/sandbox/scRNA/DE/scRNA_vs_DBiT_per_cluster_norm_log2.csv')
raw_gene_counts.to_csv('/Genomics/chanlab/blaw/Spatial_Lung/sandbox/scRNA/DE/scRNA_vs_DBiT/scRNA_vs_DBiT_per_cluster_raw_counts.csv')

In [ ]:
gene_expr = pd.read_csv('/Genomics/chanlab/blaw/Spatial_Lung/sandbox/scRNA/DE/scRNA_vs_DBiT/scRNA_vs_DBiT_per_cluster_norm_log2.csv', index_col = 0)
raw_gene_counts = pd.read_csv('/Genomics/chanlab/blaw/Spatial_Lung/sandbox/scRNA/DE/scRNA_vs_DBiT/scRNA_vs_DBiT_per_cluster_raw_counts.csv', index_col = 0)

In [ ]:
for state in cellstates:
    print(state)
    slope, intercept, r, p, se = stats.linregress(gene_expr[state + '_scRNA'], gene_expr[state + '_DBiT'])
    print('\t{}\t{}'.format('slope', slope))
    print('\t{}\t{}'.format('r_value', r))
    print('\t{}\t{}'.format('p_value', p))

In [ ]:
# Try correlation after kicking out the genes that are 0 in scRNA or DBiT
for state in cellstates:
    print(state)
    slope, intercept, r, p, se = stats.linregress(gene_expr[state + '_scRNA'], gene_expr[state + '_DBiT'])
    print('\t{}\t{}'.format('slope', slope))
    print('\t{}\t{}'.format('r_value', r))
    print('\t{}\t{}'.format('p_value', p))

In [ ]:
for state in cellstates:
    plt.plot(gene_expr['{}_scRNA'.format(state)], gene_expr['{}_DBiT'.format(state)], '.', alpha = 0.1)
    plt.xlabel('scRNA')
    plt.ylabel('DBiT')
    plt.title(state + '_Norm_Expr')
    plt.show()
    plt.close()

In [ ]:
for state in cellstates:
    plt.plot(raw_gene_counts['{}_scRNA'.format(state)], raw_gene_counts['{}_DBiT'.format(state)], '.', alpha = 0.1)
    plt.xlabel('scRNA')
    plt.ylabel('DBiT')
    plt.title(state + '_Raw_Counts')
    plt.show()
    plt.close()

In [ ]:
gene_expr

In [ ]:
weird_genes = []
for gene in gene_expr.index:
    mes1_diff = gene_expr.loc[gene, 'SyntheticMesenchyme_scRNA'] / gene_expr.loc[gene, 'SyntheticMesenchyme_DBiT']
    epi_diff = gene_expr.loc[gene, 'Epithelial(Distal)_scRNA'] / gene_expr.loc[gene, 'Epithelial(Distal)_DBiT']

    if mes1_diff - epi_diff > 1.5:
        weird_genes.append(gene)

In [ ]:
# Find the number of genes that are 0 in one but not the other dataset
weird_genes_df = pd.DataFrame(index = cellstates, columns = ['total', 'scRNA_0', 'DBiT_0'])
weird_genes_dict = {}

for cellstate in cellstates:
    dbit_0_genes = []
    scRNA_0_genes = []
    for gene in gene_expr.index:
        scRNA_temp = gene_expr.loc[gene, cellstate + '_scRNA']
        dbit_temp = gene_expr.loc[gene, cellstate + '_DBiT']
        
        if dbit_temp > 0 and scRNA_temp == 0:
            scRNA_0_genes.append(gene)
        elif dbit_temp == 0 and scRNA_temp > 0:
            dbit_0_genes.append(gene)
            
    weird_genes_dict[cellstate + '_scRNA_0'] = scRNA_0_genes
    weird_genes_dict[cellstate + '_DBiT_0'] = dbit_0_genes
            
    weird_genes_df.loc[cellstate, 'total'] = len(gene_expr.index)
    weird_genes_df.loc[cellstate, 'scRNA_0'] = len(scRNA_0_genes)
    weird_genes_df.loc[cellstate, 'DBiT_0'] = len(dbit_0_genes)
        

In [ ]:
weird_genes_df

In [ ]:
total_scRNA_set = set()
for cellstate in cellstates:
    for gene in weird_genes_dict[cellstate + '_scRNA_0']:
        total_scRNA_set.add(gene)

In [ ]:
weird_genes_df

In [ ]:
scRNA_overlap_df = pd.DataFrame(index = cellstates, columns = cellstates)

for cellstate_1 in cellstates:
    for cellstate_2 in cellstates:
        cellstate_1_genes = weird_genes_dict[cellstate_1 + '_scRNA_0']
        cellstate_2_genes = weird_genes_dict[cellstate_2 + '_scRNA_0']
        
        overlap = 0
        for gene in cellstate_1_genes:
            if gene in cellstate_2_genes:
                overlap += 1
                
        scRNA_overlap_df.loc[cellstate_1, cellstate_2] = overlap
        
for col in scRNA_overlap_df.columns:
    scRNA_overlap_df[col] = scRNA_overlap_df[col].astype('float')

# Normalize to the number of 0 genes in the row comparison
scRNA_overlap_norm = scRNA_overlap_df.copy()
for row in scRNA_overlap_norm.index:
    row_val = weird_genes_df.loc[row, 'scRNA_0']
    
    for col in scRNA_overlap_norm.columns:
        scRNA_overlap_norm.loc[row, col] = scRNA_overlap_df.loc[row, col] / row_val

sns.clustermap(scRNA_overlap_df,  row_cluster = False, col_cluster = False)
plt.title('scRNA == 0')
plt.show()
plt.close()

In [ ]:
dbit_overlap_df = pd.DataFrame(index = cellstates, columns = cellstates)

for cellstate_1 in cellstates:
    for cellstate_2 in cellstates:
        cellstate_1_genes = weird_genes_dict[cellstate_1 + '_DBiT_0']
        cellstate_2_genes = weird_genes_dict[cellstate_2 + '_DBiT_0']
        
        overlap = 0
        for gene in cellstate_1_genes:
            if gene in cellstate_2_genes:
                overlap += 1
                
        dbit_overlap_df.loc[cellstate_1, cellstate_2] = overlap
        
for col in dbit_overlap_df.columns:
    dbit_overlap_df[col] = dbit_overlap_df[col].astype('float')
    
dbit_overlap_norm = dbit_overlap_df.copy()
for row in dbit_overlap_norm.index:
    row_val = weird_genes_df.loc[row, 'DBiT_0']
    
    for col in dbit_overlap_norm.columns:
        dbit_overlap_norm.loc[row, col] = dbit_overlap_df.loc[row, col] / row_val

sns.clustermap(dbit_overlap_df,  row_cluster = False, col_cluster = False)
plt.title('DBiT == 0')
plt.show()
plt.close()

In [ ]:
scRNA_dbit_overlap_df = pd.DataFrame(index = ['DBiT0_' + i for i in cellstates], columns = ['scRNA0_' + i for i in cellstates])

for cellstate_1 in cellstates:
    for cellstate_2 in cellstates:
        cellstate_1_genes = weird_genes_dict[cellstate_1 + '_DBiT_0']
        cellstate_2_genes = weird_genes_dict[cellstate_2 + '_scRNA_0']
        
        overlap = 0
        for gene in cellstate_1_genes:
            if gene in cellstate_2_genes:
                overlap += 1
                
        scRNA_dbit_overlap_df.loc['DBiT_' + cellstate_1, 'scRNA_' + cellstate_2] = overlap
        
for col in scRNA_dbit_overlap_df.columns:
    scRNA_dbit_overlap_df[col] = scRNA_dbit_overlap_df[col].astype('float')
    
scRNA_dbit_overlap_norm = scRNA_dbit_overlap_df.copy()
for row in scRNA_dbit_overlap_norm.index:
    row_val = weird_genes_df.loc[row.split('_')[1], 'DBiT_0']
    
    for col in scRNA_dbit_overlap_norm.columns:
        scRNA_dbit_overlap_norm.loc[row, col] = scRNA_dbit_overlap_df.loc[row, col] / row_val

sns.clustermap(scRNA_dbit_overlap_df.drop(columns = ['scRNA_Unknown'], index = ['DBiT_Unknown']),  row_cluster = False, col_cluster = False)
plt.title('DBiT vs ScRNA')
plt.show()
plt.close()

In [ ]:
scRNA_set = set()
DBIT_set = set()

for cellstate in cellstates:
    for gene in weird_genes_dict[cellstate + '_scRNA_0']:
        scRNA_set.add(gene)
    for gene in weird_genes_dict[cellstate + '_DBiT_0']:
        DBIT_set.add(gene)

In [ ]:
len(scRNA_set)

In [ ]:
len(DBIT_set)

In [ ]:
overlap = 0
for gene in scRNA_set:
    if gene in DBIT_set:
        overlap += 1

In [ ]:
scRNA_0_all = []
for gene in gene_expr[[i + '_scRNA' for i in cellstates]].index:
    if gene_expr[[i + '_scRNA' for i in cellstates]].loc[gene].sum() == 0:
        scRNA_0_all.append(gene)

In [ ]:
dbit_0_all = []
for gene in gene_expr[[i + '_DBiT' for i in cellstates]].index:
    if gene_expr[[i + '_DBiT' for i in cellstates]].loc[gene].sum() == 0:
        dbit_0_all.append(gene)

In [ ]:
len(scRNA_0_all)

In [ ]:
len(dbit_0_all)

In [ ]:
for gene in gene_expr.loc[scRNA_0_all].sort_values('SmoothMuscle_DBiT', ascending = False).index:
    if gene.startswith('ENS'):
        pass
    else:
        print(gene)

In [ ]:
dbit_0_all

In [ ]:
overlap = 0
for gene in scRNA_0_all:
    if gene in dbit_0_all:
        overlap += 1
print(overlap)

In [ ]:
bad_genes = dbit_0_all + scRNA_0_all

In [ ]:
# Try correlation after kicking out the genes that are 0 in scRNA or DBiT
temp_gene_expr = gene_expr.copy()
temp_gene_expr.drop(index = bad_genes, inplace = True)
for state in cellstates:
    print(state)
    slope, intercept, r, p, se = stats.linregress(temp_gene_expr[state + '_scRNA'], temp_gene_expr[state + '_DBiT'])
    print('\t{}\t{}'.format('slope', slope))
    print('\t{}\t{}'.format('r_value', r))
    print('\t{}\t{}'.format('p_value', p))

In [ ]:
apoptosis_genes = ['Tnfr', 'Bh3', 'Bcl2', 'Bax', 'Bak', 'Smac', 'Htra2', 'Arts', 'Fadd', 'CytC', 'Apf1', 'Casp8', 'Casp9', 'Casp7', 'Casp3', 'Xiap', '']

In [ ]:
good_apoptosis_genes = []
for i in lung_adata.var_names:
    if i in apoptosis_genes:
        good_apoptosis_genes.append(i)

In [ ]:
good_apoptosis_genes

In [ ]:
sc.tl.score_genes(lung_adata, good_apoptosis_genes, ctrl_size = len(good_apoptosis_genes), score_name = 'module_apoptosis_score', use_raw = False)

In [ ]:
for time in timepoints:
    datasets = lung_adata.obs[(lung_adata.obs['timepoint'] == time) & (lung_adata.obs['tech'] == 'DBiT_seq')]['orig.ident'].unique()
    fig, axs = plt.subplots(1, len(datasets), figsize = (6*len(datasets), 6))
    
    count = 0
    for exp in datasets:
        temp_ax = axs[count]
        count += 1

        temp_locs = lung_adata.obs[(lung_adata.obs['orig.ident'] == exp)].index

        x = [0]
        y = [0]
        z = [0]
        s = [20]
        for i in temp_locs:
            z.append(lung_adata.obs.loc[i, 'module_apoptosis_score'])
            y.append(int(i.split('_')[0].split('x')[0]))
            x.append(int(i.split('_')[0].split('x')[1]))
            s.append(20)
        temp_ax.scatter(x, y, c = z, cmap = 'viridis')
        temp_ax.set_title('{}_Apoptosis_Score'.format(exp))
        temp_ax.set_xlabel('X Pos')
        temp_ax.set_ylabel('Y Pos')
        temp_ax.set_xlim(0, 52)
        temp_ax.set_ylim(0, 52)
    plt.show()
    plt.close()

In [ ]:
data = []

for i in cellstates:
    data.append(lung_adata.obs[lung_adata.obs['cellstate'] == i]['module_apoptosis_score'])

fig, ax = plt.subplots(figsize = (10, 5))

plt.boxplot(data, labels = cellstates)
plt.title('Module Score - Apoptosis')
plt.ylabel('Module Score')
plt.xticks(rotation = 45)
plt.tight_layout()
plt.show()
plt.close()